# EXH MMM — Budget Scenario Analysis & Stakeholder Deliverable (FINAL)
**Three annual scenarios on the validated grouped model, with posterior credible intervals.**

Operates on the grouped model's **validated** group saturation curves (Val R² 0.484). The decision
variable is **group-level spend**; LOB-level outcomes follow from the graft's tilt weights. This is
the faithful, model-backed level — it does not invent within-group LOB response curves.

### Scenarios (annual outcomes only)
1. **Budget uplift** — total 2025 spend ×(1.10, 1.20, 1.30), spend *pattern unchanged*, baseline
   unchanged. Shows attendance lift; saturation means lift < spend increase.
2. **Reshuffle** — siphon up to 10% from **low-marginal** groups, redistribute to **reallocatable
   (YES) groups with curve headroom**. Total budget constant. Directional/frozen/geo-test groups
   can be siphoned from but never receive.
3. **Combined** — uplift × reshuffle together.

### Uncertainty
Each scenario reports the predicted annual attendance lift with an **89% credible interval** from
the grouped model's posterior. The CI is labeled honestly: *"89% probability the true lift is in
this range, conditional on the model"* — **not** the probability the scenario is enacted (that is a
business decision the model has no view on). A High/Medium/Low confidence tag is derived from CI
width.

In [0]:
%pip install arviz h5netcdf netcdf4 matplotlib --quiet

In [0]:
import os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az

GROUPED_PATH = "/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/MMM_EXH_Outcomes_v2/"
DECOMP_PATH  = "/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/MMM_EXH_LOB_Decomp_v2/"
OUT_PATH     = DECOMP_PATH

with open(f"{GROUPED_PATH}response_params.pkl",       "rb") as f: response_params       = pickle.load(f)
with open(f"{GROUPED_PATH}spend_train_max_iter2.pkl", "rb") as f: spend_train_max_iter2 = pickle.load(f)
with open(f"{GROUPED_PATH}model_meta.pkl",            "rb") as f: model_meta            = pickle.load(f)
with open(f"{GROUPED_PATH}scaler_y.pkl",              "rb") as f: scaler_y              = pickle.load(f)
with open(f"{DECOMP_PATH}lob_decomposition.pkl",      "rb") as f: decomp_artifact       = pickle.load(f)

ACTIVE_COMBOS  = model_meta["ACTIVE_COMBOS"]
LOB_GROUP_MAP  = model_meta["LOB_GROUP_MAP"]
scale_factor   = float(scaler_y.scale_[0])

# Derive baseline_2025 from df_full (v2 pipeline doesn't export baseline_2025.pkl separately)
df_full = pd.read_parquet(f"{GROUPED_PATH}df_full.parquet")
df_full["year"] = pd.to_datetime(df_full["week_date"]).dt.year
_funnels = ["Awareness", "Consideration", "Conversion"]
_media_cols = [c for c in df_full.columns if any(c.startswith(f"{f}_") for f in _funnels)
               and "__" not in c and "media_" not in c]

def _combo_cols(combo):
    funnel, group = combo.split("__", 1)
    lobs = LOB_GROUP_MAP.get(group, [])
    return [c for c in _media_cols if c.startswith(f"{funnel}_")
            and any(lob in c for lob in lobs)]

baseline_2025 = {}
_df_2025 = df_full[df_full["year"] == 2025]
for combo in ACTIVE_COMBOS:
    cols = _combo_cols(combo)
    weekly_spend = _df_2025[cols].sum(axis=1)
    annual_spend = float(weekly_spend.sum())
    weeks_active = int((weekly_spend > 0).sum())
    weekly_avg   = annual_spend / 52
    baseline_2025[combo] = {
        "annual_spend": annual_spend,
        "weekly_avg": weekly_avg,
        "weeks_active": weeks_active,
    }
del _df_2025  # free temp slice, keep df_full for downstream cells
decomp_table  = decomp_artifact["decomp_table"]
reallocatable_lobs = set(decomp_artifact["reallocatable_lobs"])

# Map each combo -> set of reallocatable LOBs it contains (for give-eligibility)
combo_has_realloc = {}
for combo in ACTIVE_COMBOS:
    members = decomp_table[(decomp_table["Parent Combo"] == combo)]
    combo_has_realloc[combo] = bool((members["Reallocatable"] == "YES").any())

CPX_NAVY="#002B5C"; CPX_RED="#E31837"; CPX_GOLD="#F5A800"; CPX_BLUE="#0857C3"; CPX_GREEN="#2E8B57"
plt.rcParams.update({"font.family":"sans-serif","font.sans-serif":["Arial","DejaVu Sans"],
    "axes.spines.top":False,"axes.spines.right":False,"figure.facecolor":"white","figure.dpi":120})

print(f"Loaded {len(ACTIVE_COMBOS)} validated group curves")
print(f"Reallocatable (can receive) combos: {[c for c in ACTIVE_COMBOS if combo_has_realloc[c]]}")

## 1. Group Response Curve & Contribution Functions
Annual contribution for a combo at a given annual spend, on its validated saturation curve.

In [0]:
def combo_contribution(combo, annual_spend, params=None):
    p  = params or response_params[combo]
    mx = spend_train_max_iter2[combo]
    s  = (annual_spend / 52) / mx
    ad = s / (1 - p["alpha"])                       # steady-state adstock
    if p["saturation"] == "hill":
        sat = (ad ** p["alpha_s"]) / (ad ** p["alpha_s"] + p["k"] ** p["alpha_s"])
    else:
        sat = 1 - np.exp(-p["lam"] * ad)
    return p["gamma"] * sat * scale_factor * 52

def marginal_return(combo, annual_spend, delta=5200.0):
    up = combo_contribution(combo, annual_spend + delta)
    dn = combo_contribution(combo, max(0, annual_spend - delta))
    return (up - dn) / (2 * delta)

baseline_spend = {c: baseline_2025[c]["annual_spend"] for c in ACTIVE_COMBOS}
total_baseline_spend = sum(baseline_spend.values())
baseline_attendance  = sum(combo_contribution(c, baseline_spend[c]) for c in ACTIVE_COMBOS)

print(f"Baseline annual media spend     : ${total_baseline_spend:,.0f}")
print(f"Baseline annual media attendance: {baseline_attendance:,.0f}")
print("\nPer-group marginal return at baseline (att per next $):")
for c in sorted(ACTIVE_COMBOS, key=lambda x: -marginal_return(x, baseline_spend[x])):
    print(f"  {c:<40} {marginal_return(c, baseline_spend[c]):.2f}  "
          f"{'(can receive)' if combo_has_realloc[c] else '(siphon-only)'}")

## 2. Posterior CI Engine
Propagates the grouped model's posterior through any group spend allocation to get an 89% credible
interval on annual attendance. Loaded once and reused across scenarios.

In [0]:
def _load_posterior():
    """Hard requirement: the grouped model's posterior trace must be present.
    No proxy fallback — CIs are only produced from the real posterior."""
    candidates = ["trace_iter2_mixed_hier.nc", "trace.nc", "trace_grouped.nc", "trace_iter2.nc"]
    for fn in candidates:
        path = f"{GROUPED_PATH}{fn}"
        if os.path.exists(path):
            print(f"Loaded posterior: {fn}")
            return az.from_netcdf(path).posterior
    raise FileNotFoundError(
        "Grouped-model trace not found in "
        f"{GROUPED_PATH} (looked for {candidates}). "
        "CIs require the real posterior — fix the path/filename and re-run. "
        "No proxy band will be produced.")

POST = _load_posterior()

# Pre-extract flat per-draw param arrays for every combo, once.
def _flat(name, combo, default):
    key = f"{name}_{combo}"
    if key in POST:
        return POST[key].values.ravel()
    return None  # signal: param not sampled, use point estimate

N_DRAWS = 800
# size the draw pool from any available sampled param
_probe = None
for c in ACTIVE_COMBOS:
    for nm in ["gamma","lam","alpha"]:
        arr = _flat(nm, c, None)
        if arr is not None:
            _probe = arr; break
    if _probe is not None: break
if _probe is None:
    raise RuntimeError("Trace loaded but contains no recognised combo params (gamma/lam/alpha).")
_pool = _probe.shape[0]
DRAW_IDX = np.random.choice(_pool, size=min(N_DRAWS, _pool), replace=False)

# Cache per-combo per-draw param vectors
_PARAM_CACHE = {}
for c in ACTIVE_COMBOS:
    p = response_params[c]
    def pick(nm, dflt):
        arr = _flat(nm, c, None)
        return arr[DRAW_IDX] if arr is not None else np.full(len(DRAW_IDX), dflt)
    alpha_arr = pick("alpha", p["alpha"])
    alpha_arr = np.minimum(alpha_arr, 0.999)  # clamp to avoid division by zero
    entry = {"alpha": alpha_arr, "gamma": pick("gamma", p["gamma"]),
             "sat": p["saturation"], "mx": spend_train_max_iter2[c]}
    if p["saturation"] == "hill":
        entry["alpha_s"] = pick("alpha_s", p["alpha_s"]); entry["k"] = pick("k", p["k"])
    else:
        entry["lam"] = pick("lam", p["lam"])
    _PARAM_CACHE[c] = entry
print(f"Posterior ready: {len(DRAW_IDX)} draws cached for {len(ACTIVE_COMBOS)} combos.")

def combo_contribution_draws(combo, annual_spend):
    """Vector of contributions across all cached draws for one combo at one spend."""
    e = _PARAM_CACHE[combo]
    s  = (annual_spend / 52) / e["mx"]
    ad = s / (1 - e["alpha"])
    if e["sat"] == "hill":
        sat = (ad ** e["alpha_s"]) / (ad ** e["alpha_s"] + e["k"] ** e["alpha_s"])
    else:
        sat = 1 - np.exp(-e["lam"] * ad)
    return e["gamma"] * sat * scale_factor * 52   # vector over draws

def total_attendance_draws(spend_alloc):
    """Per-draw total attendance vector for an allocation."""
    tot = np.zeros(len(DRAW_IDX))
    for c, sp in spend_alloc.items():
        tot += combo_contribution_draws(c, sp)
    return tot

# Baseline per-draw vector (shared across all scenarios → enables paired differencing)
BASELINE_DRAWS = total_attendance_draws(baseline_spend)
print(f"Baseline attendance: point {baseline_attendance:,.0f} | "
      f"draw mean {BASELINE_DRAWS.mean():,.0f}")

## 3. Confidence Label from CI Width
Translates the credible-interval width into a High/Medium/Low tag. This is the honest version of a
"confidence" number — it reflects model uncertainty about the lift, **not** the probability the
scenario is enacted.

In [0]:
def lift_with_ci(scenario_alloc, label, ci=0.89):
    """Paired per-draw lift CI: (scenario_draw - baseline_draw) draw by draw.
    Shared parameter uncertainty cancels, so this reflects uncertainty in the LIFT,
    not in the absolute level."""
    scen_draws = total_attendance_draws(scenario_alloc)
    lift_draws = scen_draws - BASELINE_DRAWS              # paired difference
    pct_draws  = lift_draws / BASELINE_DRAWS * 100        # % lift per draw

    point_att = sum(combo_contribution(c, scenario_alloc[c]) for c in scenario_alloc)
    lift_pt   = point_att - baseline_attendance
    pct_pt    = lift_pt / baseline_attendance * 100

    lo_q, hi_q = (1-ci)/2, 1-(1-ci)/2
    pct_lo = float(np.quantile(pct_draws, lo_q))
    pct_hi = float(np.quantile(pct_draws, hi_q))
    width  = pct_hi - pct_lo
    conf = "High" if width < 4 else ("Medium" if width < 8 else "Low")
    crosses_zero = (pct_lo <= 0 <= pct_hi)
    effect = "effect uncertain (CI crosses 0)" if crosses_zero else "directionally positive"
    return {"label":label, "spend":sum(scenario_alloc.values()),
            "attendance":point_att, "lift":lift_pt, "lift_pct":pct_pt,
            "pct_lo":pct_lo, "pct_hi":pct_hi, "ci_width":width,
            "confidence":conf, "crosses_zero":crosses_zero, "effect":effect, "proxy":False}

print("Lift CI = 89% quantiles of the PAIRED per-draw difference (scenario - baseline).")
print("Confidence: CI width <4pp High, 4-8pp Medium, >8pp Low.")

## 4. Scenario 1 — Budget Uplift (+10 / +20 / +30%)
Spend pattern unchanged: every group scaled by the same factor. Saturation means attendance lift is
sub-proportional to the spend increase.

In [0]:
s1_results = []
for factor in [1.10, 1.20, 1.30]:
    alloc = {c: baseline_spend[c] * factor for c in ACTIVE_COMBOS}
    r = lift_with_ci(alloc, f"+{round((factor-1)*100)}% budget")
    r["spend_increase_pct"] = (factor-1)*100
    s1_results.append(r)

s1_df = pd.DataFrame([{
    "Scenario": r["label"], "Annual Spend": f"${r['spend']:,.0f}",
    "Spend ↑": f"+{r['spend_increase_pct']:.0f}%",
    "Attendance": f"{r['attendance']:,.0f}",
    "Att Lift": f"+{r['lift']:,.0f}",
    "Lift %": f"+{r['lift_pct']:.1f}%",
    "89% CI": f"[+{r['pct_lo']:.1f}%, +{r['pct_hi']:.1f}%]",
    "Confidence": r["confidence"],
} for r in s1_results])
print("SCENARIO 1 — BUDGET UPLIFT (pattern unchanged)\n")
display(s1_df)
print("\nNote: each lift % is sub-proportional to the spend increase due to saturation.")
print("CI = 89% probability the TRUE lift lies in range, conditional on the model.")

## 5. Scenario 2 — Reshuffle (capped 10% siphon → high-marginal reallocatable groups)
Siphon up to 10% from **low-marginal** groups, pool it, give to **reallocatable groups with
headroom** ranked by marginal return. Total budget held constant.

In [0]:
def reshuffle(base_alloc, siphon_cap=0.10):
    alloc = dict(base_alloc)
    # rank by marginal return at current spend
    mr = {c: marginal_return(c, alloc[c]) for c in ACTIVE_COMBOS}
    med_mr = np.median(list(mr.values()))
    # siphon from BELOW-median-marginal groups (low marginal = near saturation / inefficient)
    pool = 0.0
    for c in ACTIVE_COMBOS:
        if mr[c] < med_mr:
            take = alloc[c] * siphon_cap
            alloc[c] -= take; pool += take
    # receivers: reallocatable groups with ABOVE-median marginal, ranked desc
    receivers = sorted([c for c in ACTIVE_COMBOS if combo_has_realloc[c] and mr[c] >= med_mr],
                       key=lambda x: -mr[x])
    if not receivers:
        receivers = sorted([c for c in ACTIVE_COMBOS if combo_has_realloc[c]], key=lambda x:-mr[x])
    # distribute pool proportional to marginal return, but cap each receiver's GAIN at +10% too
    total_mr = sum(mr[c] for c in receivers)
    remaining = pool
    for c in receivers:
        want = pool * (mr[c]/total_mr) if total_mr>0 else pool/len(receivers)
        cap  = base_alloc[c] * siphon_cap
        give = min(want, cap, remaining)
        alloc[c] += give; remaining -= give
    # if cap left budget unspent, return it pro-rata to siphoned groups (keep total constant)
    if remaining > 1e-6:
        siphoned = [c for c in ACTIVE_COMBOS if mr[c] < med_mr]
        for c in siphoned:
            alloc[c] += remaining/len(siphoned)
    return alloc

s2_alloc = reshuffle(baseline_spend)
r2 = lift_with_ci(s2_alloc, "Reshuffle (10% cap)")

# Movement table
move_rows = []
for c in ACTIVE_COMBOS:
    d = s2_alloc[c]-baseline_spend[c]
    move_rows.append({"Group":c,"Baseline":f"${baseline_spend[c]:,.0f}",
        "Reshuffled":f"${s2_alloc[c]:,.0f}","Δ":f"{d:+,.0f}",
        "Δ%":f"{d/baseline_spend[c]*100:+.1f}%",
        "Role":"receiver" if d>1 else ("siphoned" if d<-1 else "—")})
print("SCENARIO 2 — RESHUFFLE\n")
display(pd.DataFrame(move_rows))
print(f"\nTotal spend: ${sum(s2_alloc.values()):,.0f} (baseline ${total_baseline_spend:,.0f}, held constant)")
print(f"Attendance: {r2['attendance']:,.0f}  Lift: +{r2['lift']:,.0f} (+{r2['lift_pct']:.1f}%)  "
      f"89% CI [+{r2['pct_lo']:.1f}%, +{r2['pct_hi']:.1f}%]  Confidence: {r2['confidence']}")

## 6. Scenario 3 — Combined (uplift + reshuffle)
Apply each budget uplift, then reshuffle the larger budget into high-marginal reallocatable groups
(same 10% cap on moves).

In [0]:
s3_results = []
for factor in [1.10, 1.20, 1.30]:
    uplifted = {c: baseline_spend[c]*factor for c in ACTIVE_COMBOS}
    combined = reshuffle(uplifted)
    r = lift_with_ci(combined, f"+{round((factor-1)*100)}% + reshuffle")
    r["spend_increase_pct"] = (factor-1)*100
    s3_results.append(r)

s3_df = pd.DataFrame([{
    "Scenario": r["label"], "Annual Spend": f"${r['spend']:,.0f}",
    "Attendance": f"{r['attendance']:,.0f}", "Lift %": f"+{r['lift_pct']:.1f}%",
    "89% CI": f"[+{r['pct_lo']:.1f}%, +{r['pct_hi']:.1f}%]", "Confidence": r["confidence"],
} for r in s3_results])
print("SCENARIO 3 — COMBINED\n")
display(s3_df)

## 7. Annual Outcome Visuals
Three charts: (1) lift vs spend increase for S1, (2) reshuffle movement, (3) all scenarios with CIs.

In [0]:
# ── CELL: Scenario outcome visuals — CPX brand style ─────────────────────────

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

CPX_NAVY  = "#002B5C"
CPX_RED   = "#E31837"
CPX_GOLD  = "#F5A800"
CPX_GREEN = "#2E8B57"
CPX_GREY  = "#9AA0A6"
CPX_DGREY = "#4A4A4A"
CPX_LGREY = "#CCCCCC"

def save_fig(fig, fname):
    fig.savefig(os.path.join(OUT_PATH, fname), dpi=150, bbox_inches="tight", facecolor="white")
    print(f"Saved: {fname}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# ── (A) S1: Diminishing returns curve ────────────────────────────────────────
ax = axes[0]
sp = [0] + [r["spend_increase_pct"] for r in s1_results]
at = [0] + [r["lift_pct"]           for r in s1_results]

ax.plot(sp, sp, "--", color=CPX_LGREY, lw=1.5, label="1:1 (no saturation)")
ax.plot(sp, at, "-o", color=CPX_NAVY,  lw=2.5, markersize=8, label="Predicted attendance lift")

for r in s1_results:
    ax.errorbar(r["spend_increase_pct"], r["lift_pct"],
                yerr=[[r["lift_pct"] - r["pct_lo"]], [r["pct_hi"] - r["lift_pct"]]],
                color=CPX_NAVY, capsize=5, alpha=0.5)
    ax.annotate(f"+{r['lift_pct']:.1f}%",
                (r["spend_increase_pct"], r["lift_pct"]),
                textcoords="offset points", xytext=(8, -4),
                fontsize=9, fontweight="bold", color=CPX_NAVY)

ax.set_xlabel("Budget increase (%)", fontsize=9, color=CPX_DGREY)
ax.set_ylabel("Attendance lift (%)",  fontsize=9, color=CPX_DGREY)
ax.set_title("Scenario 1 — Diminishing Returns on Budget Uplift",
             fontweight="bold", color=CPX_NAVY, fontsize=11)
ax.legend(frameon=False, fontsize=9)
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.2, color=CPX_LGREY)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(labelsize=8)

# ── (B) All scenarios — bar chart with CI whiskers ────────────────────────────
ax = axes[1]
all_results = s1_results + [r2] + s3_results
labels  = [r["label"]    for r in all_results]
lifts   = [r["lift_pct"] for r in all_results]
los     = [r["lift_pct"] - r["pct_lo"]  for r in all_results]
his     = [r["pct_hi"]   - r["lift_pct"] for r in all_results]
colors  = [CPX_NAVY] * 3 + [CPX_GOLD] + [CPX_GREEN] * 3
x       = np.arange(len(labels))

ax.bar(x, lifts, color=colors, alpha=0.88, edgecolor="white")
ax.errorbar(x, lifts, yerr=[los, his],
            fmt="none", ecolor=CPX_DGREY, capsize=4, alpha=0.6)

for i, r in enumerate(all_results):
    tag   = "effect uncertain" if r["crosses_zero"] else r["confidence"]
    color = CPX_RED if r["crosses_zero"] else CPX_DGREY
    ax.text(i, lifts[i] + his[i] + max(his) * 0.12,
            tag, ha="center", fontsize=8, fontweight="bold", color=color)
    ax.text(i, lifts[i] / 2,
            f"+{lifts[i]:.1f}%", ha="center", fontsize=8,
            fontweight="bold", color="white")

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=8, color=CPX_DGREY)
ax.set_ylabel("Attendance lift (%)", fontsize=9, color=CPX_DGREY)
ax.set_title("All Scenarios — Annual Lift with 89% CI\n(label = model confidence from CI width)",
             fontweight="bold", color=CPX_NAVY, fontsize=11)
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.2, color=CPX_LGREY)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(labelsize=8)

# Shared legend
leg = [
    mpatches.Patch(facecolor=CPX_NAVY,  label="Budget uplift"),
    mpatches.Patch(facecolor=CPX_GOLD,  label="Reshuffle"),
    mpatches.Patch(facecolor=CPX_GREEN, label="Combined"),
]
fig.legend(handles=leg, loc="lower center", ncol=3, frameon=False,
           fontsize=9, bbox_to_anchor=(0.5, -0.06))

fig.suptitle("EXH MMM — Budget Scenario Outcomes (2025)",
             fontsize=13, fontweight="bold", color=CPX_NAVY, y=1.02)

fig.text(0.99, -0.04,
         "Cineplex Marketing Analytics  |  EXH MMM Phase 2 Iter 2",
         ha="right", va="bottom", fontsize=6, color=CPX_LGREY)

plt.tight_layout()
save_fig(fig, "scenario_outcomes.png")
plt.show()

print("\nCIs are paired per-draw lift intervals from the grouped posterior —")
print("tight because shared parameter uncertainty cancels in the scenario-minus-baseline difference.")

In [0]:
# ── Load decomp artifact if not already in scope ─────────────────────────────
if "decomp" not in dir():
    with open(f"{DECOMP_PATH}lob_decomposition.pkl", "rb") as f:
        _da = pickle.load(f)
    decomp = _da["decomp_table"]
    print("Loaded decomp table from pickle.")
else:
    print("Using decomp already in scope.")

In [0]:
# ── CELL: Per-Scenario LOB Spend — Baseline vs Scenario (graft-split) ─────────
# For each scenario: split group-level spend to LOBs using graft Final Share (grp),
# then plot baseline vs scenario LOB spend side by side, coloured by funnel.

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
import os

CPX_NAVY  = "#002B5C"
CPX_RED   = "#E31837"
CPX_GOLD  = "#F5A800"
CPX_DGREY = "#4A4A4A"
CPX_LGREY = "#CCCCCC"

FUNNEL_COLORS = {"Awareness": CPX_NAVY, "Consideration": CPX_RED, "Conversion": CPX_GOLD}

# ── Construct funnel_lob_cols from df_full column names ───────────────────────
_CHANNELS = ["Digital", "Social", "SEM"]
_FUNNELS  = ["Awareness", "Consideration", "Conversion"]
funnel_lob_cols = {}
for col in df_full.columns:
    if "__" in col or col.startswith("media_"):
        continue
    for f in _FUNNELS:
        if col.startswith(f + "_"):
            suffix = col[len(f)+1:]  # e.g. "Cineclub_Digital"
            for ch in _CHANNELS:
                if suffix.endswith("_" + ch):
                    lob_raw = suffix[:-(len(ch)+1)]  # e.g. "Cineclub"
                    funnel_lob_cols.setdefault((f, lob_raw), []).append(col)
                    break
            break

def lob_display_name(lob_raw):
    """Identity — raw LOB names already match decomp table display names."""
    return lob_raw

def save_fig(fig, fname):
    fig.savefig(os.path.join(OUT_PATH, fname), dpi=150, bbox_inches="tight", facecolor="white")
    print(f"Saved: {fname}")

# ── 1. Build LOB share table from graft decomp ───────────────────────────────
decomp_valid = decomp[decomp["Status"].str.startswith("Decomposed")].copy()
decomp_valid["share_frac"] = decomp_valid["Final Share (grp)"] / 100.0

# ── 2. split_to_lobs — validated LOBs only (via graft shares) ────────────────
def split_to_lobs(group_alloc):
    lob_spend = {}
    for _, row in decomp_valid.iterrows():
        combo    = row["Parent Combo"]
        funnel   = row["Funnel"]
        lob      = row["LOB"]
        share    = row["share_frac"]
        group_sp = group_alloc.get(combo, 0.0)
        key      = (funnel, lob)
        lob_spend[key] = lob_spend.get(key, 0.0) + group_sp * share
    return lob_spend

# ── 2b. All-LOB spend — validated via graft + unvalidated held fixed ──────────
years_avail_s = sorted(y for y in df_full["year"].dropna().unique().astype(int) if y < 2026)

def all_lob_spend(group_alloc):
    """
    Returns spend per (funnel, lob) for ALL original LOBs.
    Validated: group_alloc × graft share.
    Unvalidated/geo-test: 2025 actual spend held fixed (scale=1.0).
    """
    # Start with validated split
    lob_spend = split_to_lobs(group_alloc)

    # Add unvalidated LOBs at fixed 2025 spend
    for (funnel, lob_raw), cols in funnel_lob_cols.items():
        disp  = lob_display_name(lob_raw)
        key   = (funnel, disp)
        if key in lob_spend:
            continue  # already covered by graft
        sp_2025 = float(df_full[df_full["year"] == 2025][cols].sum().sum())
        if sp_2025 > 0:
            lob_spend[key] = sp_2025  # held fixed

    return lob_spend

baseline_lob_spend = all_lob_spend(baseline_spend)

# ── 3. Scenarios ──────────────────────────────────────────────────────────────
scenarios = (
    [(r["label"], {c: baseline_spend[c] * (1 + r["spend_increase_pct"]/100)
                   for c in ACTIVE_COMBOS})
     for r in s1_results]
    + [("Reshuffle (10% cap)", s2_alloc)]
    + [(r["label"], {c: reshuffle({c2: baseline_spend[c2] * (1 + r["spend_increase_pct"]/100)
                                   for c2 in ACTIVE_COMBOS})[c]
                    for c in ACTIVE_COMBOS})
       for r in s3_results]
)

# ── 4. LOB ordering — by baseline spend descending ───────────────────────────
# Flag which LOBs are unvalidated (held fixed)
validated_keys = {(row["Funnel"], row["LOB"]) for _, row in decomp_valid.iterrows()}

lob_keys = sorted(baseline_lob_spend.keys(),
                  key=lambda k: -baseline_lob_spend[k])
n_lobs   = len(lob_keys)
x        = np.arange(n_lobs)
bar_w    = 0.35

# ── 5. One chart per scenario ─────────────────────────────────────────────────
for scen_label, group_alloc in scenarios:
    scen_lob_spend = all_lob_spend(group_alloc)

    fig, ax = plt.subplots(figsize=(16, 9))
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.2, color=CPX_LGREY)

    for i, (funnel, lob) in enumerate(lob_keys):
        base_val = baseline_lob_spend.get((funnel, lob), 0.0) / 1e6
        scen_val = scen_lob_spend.get((funnel, lob), 0.0)     / 1e6
        color    = FUNNEL_COLORS[funnel]

        # Baseline bar — full opacity
        ax.bar(x[i] - bar_w / 2, base_val, bar_w * 0.88,
               color=color, alpha=0.9, edgecolor="white", lw=0.4)

        # Scenario bar — hatched to distinguish
        ax.bar(x[i] + bar_w / 2, scen_val, bar_w * 0.88,
               color=color, alpha=0.5, edgecolor="white", lw=0.4, hatch="///")

        # Delta label above the taller bar
        delta = scen_val - base_val
        top   = max(base_val, scen_val)
        is_validated = (funnel, lob) in validated_keys
        if abs(delta) > 0.005:
            ax.annotate(f"{delta:+.2f}M",
                        xy=(x[i] + bar_w / 2, top),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=9, fontweight="bold",
                        color=CPX_NAVY if delta > 0 else CPX_RED)
        elif not is_validated:
            ax.annotate("— fixed",
                        xy=(x[i], top),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=7, color=CPX_LGREY,
                        fontweight="bold")
        # Attendance lift label — top right of each chart
        lift_r = next(r for r in (s1_results + [r2] + s3_results) if r["label"] == scen_label)
        lift_text = (
            f"Attendance Lift: +{lift_r['lift_pct']:.1f}%\n"
            f"89% CI: [+{lift_r['pct_lo']:.1f}%, +{lift_r['pct_hi']:.1f}%]\n"
            f"Confidence: {lift_r['confidence']}\n"
            f"— = spend held fixed (geo-test or unvalidated LOB)"
        )

    ax.text(
        0.99, 0.72, lift_text,
        transform=ax.transAxes,
        ha="right", va="top",
        fontsize=11, color=CPX_DGREY, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                  edgecolor=CPX_LGREY, alpha=0.95),
    )

    # X-axis — LOB labels
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{lob}\n({fun[:3]})" for fun, lob in lob_keys],
        rotation=45, ha="right", fontsize=10, color=CPX_DGREY
    )

    ax.set_ylabel("Annual Media Spend ($M)", fontsize=11, color=CPX_DGREY)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.1f}M"))
    ax.tick_params(axis="x", labelsize=10)
    ax.set_ylim(bottom=0, top=ax.get_ylim()[1] * 1.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="y", labelsize=10)

    # Legend: funnel colours + baseline vs scenario
    funnel_patches = [mpatches.Patch(color=FUNNEL_COLORS[f], label=f) for f in FUNNEL_COLORS]
    style_patches  = [
        mpatches.Patch(facecolor=CPX_DGREY, alpha=0.9, label="Baseline 2025"),
        mpatches.Patch(facecolor=CPX_DGREY, alpha=0.5, hatch="///", label="Scenario"),
    ]
    ax.legend(handles=funnel_patches + style_patches,
              fontsize=10, loc="upper right",
              framealpha=0.95, edgecolor=CPX_LGREY)

    ax.set_title(
        f"LOB Spend — Baseline vs {scen_label}\nSplit via graft tilt shares  |  Δ labels in $M",
        fontsize=13, fontweight="bold", color=CPX_NAVY, pad=10
    )

    fig.text(0.99, 0.01,
             "Cineplex Marketing Analytics  |  EXH MMM Phase 2 Iter 2",
             ha="right", va="bottom", fontsize=6, color=CPX_LGREY)

    plt.tight_layout()

    safe = scen_label.replace(" ", "_").replace("%", "pct").replace("+", "plus").replace("/", "_")
    save_fig(fig, f"I2_B10_LOB_Spend_{safe}.png")
    plt.show()
    plt.close(fig)

In [0]:
# Master summary table + export
summary = pd.DataFrame([{
    "Scenario": r["label"],
    "Annual Spend": round(r["spend"],0),
    "Annual Attendance": round(r["attendance"],0),
    "Lift vs Baseline": round(r["lift"],0),
    "Lift %": round(r["lift_pct"],1),
    "CI Low %": round(r["pct_lo"],1), "CI High %": round(r["pct_hi"],1),
    "Confidence": r["confidence"],
} for r in (s1_results+[r2]+s3_results)])
display(summary)
summary.to_csv(f"{OUT_PATH}scenario_summary.csv", index=False)
print(f"\n✓ Saved scenario_summary.csv")
print("\nReminder: 'Confidence' and the CI reflect MODEL uncertainty about the lift —")
print("they are NOT the probability a scenario is enacted (that is a business decision).")

---
# Stakeholder Deliverable — Finalized Scenario Outcomes

This section consolidates the analysis into presentation-ready outputs:
* **Both frames side by side** — media-attributable lift (what the model moves) *and* total-attendance
  lift (what it means for real admissions), so the compression is explicit.
* **Honest reshuffle labeling** — flagged as directional/inconclusive where its CI crosses zero.
* **Exported tables** — formatted Excel (`.xlsx`) with labeled headers + CSV backup.

All numbers reconcile to **actual 2025 total attendance (42,511,640)**. Media drives ~23% of total,
so media lifts compress to smaller total-attendance lifts. The ~1.68M geo-test contribution is
contested attribution *inside* the total (hypothesis, requires a test), never added on top.

In [0]:
# ── Reconciliation inputs (from df_full, computed earlier) ────────────────────
df25 = df_full[df_full["year"] == 2025] if "df_full" in dir() else None
import pandas as pd, numpy as np, pickle, os

if df25 is None:
    df_full = pd.read_parquet(f"{GROUPED_PATH}df_full.parquet")
    df25 = df_full[df_full["year"] == 2025]

actual_total  = df25[model_meta["target_col"]].sum()
pred_total    = df25["pred"].sum()
controls_2025 = df25["controls_pred"].sum()
media_2025    = df25["total_media"].sum()
recon         = actual_total / pred_total
media_base    = media_2025    * recon
controls_base = controls_2025 * recon

# Geo-test contested attribution from the graft (extrapolated rows only)
with open(f"{DECOMP_PATH}lob_decomposition.pkl", "rb") as f: _da = pickle.load(f)
_dt = _da["decomp_table"]
geo_contested = float(_dt[_dt["Status"].str.startswith("Extrapolated")]["Annual Contrib"].dropna().sum())

MEDIA_SHARE = media_base / actual_total

# ── Scenario lift %s (media-attributable) with 89% CI, from this notebook's runs ──
scen_defs = [
    ("+10% budget",          7.1, 1.6, 8.0,  "budget"),
    ("+20% budget",         14.0, 3.0, 15.8, "budget"),
    ("+30% budget",         20.5, 4.3, 23.4, "budget"),
    ("Reshuffle (10% cap)",  1.8,-1.6, 4.2,  "reshuffle"),
    ("+10% + reshuffle",     9.1, 1.8, 11.3, "combined"),
    ("+20% + reshuffle",    16.0, 3.6, 18.9, "combined"),
    ("+30% + reshuffle",    22.6, 5.0, 26.5, "combined"),
]

def build_rows():
    rows=[]
    for name, pt, lo, hi, kind in scen_defs:
        # media-frame absolute (on reconciled validated media base)
        m_pt, m_lo, m_hi = media_base*pt/100, media_base*lo/100, media_base*hi/100
        # total-frame: admissions gained = media gain; % of real total
        t_pt = m_pt/actual_total*100; t_lo = m_lo/actual_total*100; t_hi = m_hi/actual_total*100
        crosses = lo <= 0 <= hi
        width = hi - lo
        conf = ("Directional — CI crosses 0" if crosses
                else ("High" if width<4 else ("Medium" if width<8 else "Low")))
        rows.append({
            "Scenario": name, "Type": kind,
            "Media Lift %": pt, "Media CI Low %": lo, "Media CI High %": hi,
            "Admissions Gained": m_pt, "Gained CI Low": m_lo, "Gained CI High": m_hi,
            "New Total Attendance": actual_total + m_pt,
            "Total Lift %": t_pt, "Total CI Low %": t_lo, "Total CI High %": t_hi,
            "Confidence": conf,
        })
    return pd.DataFrame(rows)

final_df = build_rows()
print(f"Reconciled: total {actual_total:,.0f} | media base {media_base:,.0f} "
      f"({MEDIA_SHARE*100:.1f}% of total) | geo-test contested {geo_contested:,.0f}")
print(f"Built {len(final_df)} scenarios.")

## Side-by-Side Visual — Media Lift vs Total-Attendance Lift
Left: lift on media-attributable attendance (what the scenarios move). Right: the same scenarios
translated to % of real total attendance. The right bars are smaller — media is ~23% of total.

In [0]:
import matplotlib.pyplot as plt
CPX_NAVY="#002B5C"; CPX_RED="#E31837"; CPX_GOLD="#F5A800"; CPX_GREEN="#2E8B57"; CPX_GREY="#9AA0A6"

def bar_color(row):
    if "crosses" in row["Confidence"]: return CPX_GREY
    return {"budget":CPX_NAVY,"combined":CPX_GREEN,"reshuffle":CPX_GOLD}[row["Type"]]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(len(final_df)); labels = final_df["Scenario"].tolist()
colors = [bar_color(r) for _, r in final_df.iterrows()]

# LEFT — media-attributable lift
ax = axes[0]
m_pt = final_df["Media Lift %"].values
m_lo = (final_df["Media Lift %"]-final_df["Media CI Low %"]).values
m_hi = (final_df["Media CI High %"]-final_df["Media Lift %"]).values
ax.bar(x, m_pt, color=colors, alpha=0.88, edgecolor="white")
ax.errorbar(x, m_pt, yerr=[m_lo,m_hi], fmt="none", ecolor="black", capsize=4, alpha=0.6)
for i,r in final_df.iterrows():
    ax.text(i, r["Media Lift %"]+m_hi[i]+0.4, f"+{r['Media Lift %']:.1f}%", ha="center",
            fontsize=8.5, fontweight="bold", color=CPX_NAVY)
ax.set_title("Media-Attributable Attendance Lift\n(what the scenarios directly move)",
             fontweight="bold", color=CPX_NAVY, fontsize=12)
ax.set_ylabel("Lift on media-attributable attendance (%)")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=8)

# RIGHT — total-attendance lift
ax = axes[1]
t_pt = final_df["Total Lift %"].values
t_lo = (final_df["Total Lift %"]-final_df["Total CI Low %"]).values
t_hi = (final_df["Total CI High %"]-final_df["Total Lift %"]).values
ax.bar(x, t_pt, color=colors, alpha=0.88, edgecolor="white")
ax.errorbar(x, t_pt, yerr=[t_lo,t_hi], fmt="none", ecolor="black", capsize=4, alpha=0.6)
ax.axhline(0, color="#333", lw=0.8)
for i,r in final_df.iterrows():
    ax.text(i, r["Total Lift %"]+t_hi[i]+0.1, f"+{r['Total Lift %']:.2f}%", ha="center",
            fontsize=8.5, fontweight="bold", color=CPX_GREEN if r["Type"]!="reshuffle" else "#8a6d00")
ax.set_title(f"Total-Attendance Lift (of {actual_total/1e6:.1f}M real admissions)\n"
             f"media is {MEDIA_SHARE*100:.0f}% of total → lifts compress",
             fontweight="bold", color=CPX_NAVY, fontsize=12)
ax.set_ylabel("Lift on TOTAL 2025 attendance (%)")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=8)

# shared legend
from matplotlib.patches import Patch
leg = [Patch(facecolor=CPX_NAVY,label="Budget uplift"), Patch(facecolor=CPX_GREEN,label="Combined"),
       Patch(facecolor=CPX_GOLD,label="Reshuffle"), Patch(facecolor=CPX_GREY,label="Directional (CI crosses 0)")]
fig.legend(handles=leg, loc="lower center", ncol=4, frameon=False, fontsize=9, bbox_to_anchor=(0.5,-0.04))
fig.suptitle("EXH MMM — Budget Scenario Outcomes (2025, reconciled to actual)",
             fontsize=14, fontweight="bold", color=CPX_NAVY, y=1.02)
plt.tight_layout()
plt.savefig(f"{DECOMP_PATH}FINAL_scenario_both_frames.png", dpi=160, bbox_inches="tight", facecolor="white")
plt.show()
print("✓ Saved FINAL_scenario_both_frames.png")

In [0]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np

CPX_NAVY  = "#002B5C"
CPX_RED   = "#E31837"
CPX_GOLD  = "#F5A800"
CPX_GREEN = "#2E8B57"
CPX_GREY  = "#9AA0A6"
CPX_DGREY = "#4A4A4A"
CPX_LGREY = "#CCCCCC"

def bar_color(row):
    if "crosses" in row["Confidence"]: return CPX_GREY
    return {"budget": CPX_NAVY, "combined": CPX_GREEN, "reshuffle": CPX_GOLD}[row["Type"]]

colors = [bar_color(r) for _, r in final_df.iterrows()]
x      = np.arange(len(final_df))
labels = final_df["Scenario"].tolist()
m_pt   = final_df["Media Lift %"].values
m_lo   = (final_df["Media Lift %"] - final_df["Media CI Low %"]).values
m_hi   = (final_df["Media CI High %"] - final_df["Media Lift %"]).values

# ═════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Media-Attributable Lift %
# ═════════════════════════════════════════════════════════════════════════════
fig1, ax = plt.subplots(figsize=(16, 9))
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.2, color=CPX_LGREY)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.bar(x, m_pt, color=colors, alpha=0.88, edgecolor="white", width=0.6)
ax.errorbar(x, m_pt, yerr=[m_lo, m_hi],
            fmt="none", ecolor=CPX_DGREY, capsize=5, alpha=0.7, lw=1.5)

for i, (_, r) in enumerate(final_df.iterrows()):
    ax.text(i, m_pt[i] * 0.5, f"+{m_pt[i]:.1f}%",
            ha="center", va="center", fontsize=10,
            fontweight="bold", color="white")
    ax.text(i, m_pt[i] + m_hi[i] + 0.3,
            f"[+{r['Media CI Low %']:.1f}%, +{r['Media CI High %']:.1f}%]",
            ha="center", va="bottom", fontsize=8,
            fontweight="bold", color=CPX_DGREY)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=9, color=CPX_DGREY)
ax.set_ylabel("Lift on media-attributable attendance (%)", fontsize=10, color=CPX_DGREY)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.set_ylim(0, max(m_pt + m_hi) * 1.45)
ax.tick_params(axis="y", labelsize=9, labelcolor=CPX_DGREY)
ax.set_title("Media-Attributable Attendance Lift (By Scenario)",
             fontweight="bold", color=CPX_NAVY, fontsize=13, pad=10)

leg1 = [
    mpatches.Patch(facecolor=CPX_NAVY,  label="Budget uplift"),
    mpatches.Patch(facecolor=CPX_GREEN, label="Combined"),
    # mpatches.Patch(facecolor=CPX_GOLD,  label="Reshuffle"),
    mpatches.Patch(facecolor=CPX_GREY,  label="Directional (CI crosses 0)"),
]
ax.legend(handles=leg1, fontsize=9, loc="upper left",
          framealpha=0.95, edgecolor=CPX_LGREY)

# fig1.text(0.99, 0.01,
#           "Cineplex Marketing Analytics  |  EXH MMM Phase 2 Iter 2",
#           ha="right", va="bottom", fontsize=6, color=CPX_LGREY)

plt.tight_layout()
plt.savefig(f"{DECOMP_PATH}FINAL_media_lift.png",
            dpi=160, bbox_inches="tight", facecolor="white")
plt.show()
print("✓ Saved FINAL_media_lift.png")

# ═════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Stacked absolute attendance: baseline + media + scenario lift
# ═════════════════════════════════════════════════════════════════════════════
non_media_base = actual_total - media_base   # baseline / non-media component

fig2, ax = plt.subplots(figsize=(16, 9))
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.2, color=CPX_LGREY)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

bar_w = 0.6
admissions_gained = final_df["Admissions Gained"].values
gained_lo = (final_df["Admissions Gained"] - final_df["Gained CI Low"]).values
gained_hi = (final_df["Gained CI High"]    - final_df["Admissions Gained"]).values

# Layer 1 — non-media baseline (grey)
ax.bar(x, non_media_base / 1e6, bar_w,
       color=CPX_GREY, alpha=0.6, edgecolor="white", lw=0.4, label="Baseline / Non-Media")

# Layer 2 — current media-attributed attendance (Navy)
ax.bar(x, media_base / 1e6, bar_w,
       bottom=non_media_base / 1e6,
       color=CPX_NAVY, alpha=0.85, edgecolor="white", lw=0.4, label="Current Media Attribution")

# Layer 3 — scenario lift (CPX Gold, consistent across all scenarios)
ax.bar(x, admissions_gained / 1e6, bar_w,
       bottom=(non_media_base + media_base) / 1e6,
       color=CPX_GOLD, alpha=0.9, edgecolor="white", lw=0.4, label="Scenario Lift")

# CI whiskers on top of scenario lift
tops = (non_media_base + media_base + admissions_gained) / 1e6
ax.errorbar(x, tops, yerr=[gained_lo / 1e6, gained_hi / 1e6],
            fmt="none", ecolor=CPX_DGREY, capsize=5, alpha=0.7, lw=1.5)

# Labels on each layer
for i, (_, r) in enumerate(final_df.iterrows()):
    # Non-media label (inside grey bar)
    ax.text(i, (non_media_base / 2) / 1e6,
            f"{non_media_base/1e6:.1f}M",
            ha="center", va="center", fontsize=8,
            fontweight="bold", color="white")

    # Media label (inside navy bar)
    ax.text(i, (non_media_base + media_base / 2) / 1e6,
            f"{media_base/1e6:.1f}M",
            ha="center", va="center", fontsize=8,
            fontweight="bold", color="white")

    # Scenario lift label (inside gold bar) — only if segment tall enough
    if admissions_gained[i] / 1e6 > 0.1:
        ax.text(i, (non_media_base + media_base + admissions_gained[i] / 2) / 1e6,
                f"+{admissions_gained[i]/1e6:.2f}M",
                ha="center", va="center", fontsize=8,
                fontweight="bold", color=CPX_NAVY)

    # CI label above whisker
    ax.text(i, tops[i] + gained_hi[i] / 1e6 + 0.1,
            f"[{r['Gained CI Low']:+,.0f}, {r['Gained CI High']:+,.0f}]",
            ha="center", va="bottom", fontsize=7.5,
            fontweight="bold", color=CPX_DGREY)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=9, color=CPX_DGREY)
ax.set_ylabel("Annual Attendance (M)", fontsize=10, color=CPX_DGREY)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}M"))
ax.set_ylim(0, (actual_total + max(admissions_gained) + max(gained_hi)) / 1e6 * 1.15)
ax.tick_params(axis="y", labelsize=9, labelcolor=CPX_DGREY)
ax.set_title(
    f"Total Attendance by Scenario  |  Baseline {actual_total/1e6:.1f}M  →  Baseline + Media + Scenario Lift",
    fontweight="bold", color=CPX_NAVY, fontsize=13, pad=10
)

ax.legend(fontsize=9, loc="upper left", framealpha=0.95, edgecolor=CPX_LGREY)

fig2.text(0.99, 0.01,
          "Cineplex Marketing Analytics  |  EXH MMM Phase 2 Iter 2",
          ha="right", va="bottom", fontsize=6, color=CPX_LGREY)

plt.tight_layout()
plt.savefig(f"{DECOMP_PATH}FINAL_attendance_stacked.png",
            dpi=160, bbox_inches="tight", facecolor="white")
plt.show()
print("✓ Saved FINAL_attendance_stacked.png")

In [0]:
# ── 2026 Forecast + Scenario Lift Projections ────────────────────────────────
# Applies each scenario's validated media lift to the 2026 forecasted attendance

FORECAST_2026 = 46_400_000  # from EXH 2026 forecast

# Build projection table
proj_rows = []
for _, r in final_df.iterrows():
    gained_pt = r["Admissions Gained"]
    gained_lo = r["Gained CI Low"]
    gained_hi = r["Gained CI High"]

    proj_rows.append({
        "Scenario"             : r["Scenario"],
        "Type"                 : r["Type"],
        "2026 Forecast (M)"    : FORECAST_2026 / 1e6,
        "Scenario Lift (M)"    : gained_pt / 1e6,
        "Projected 2026 (M)"   : (FORECAST_2026 + gained_pt) / 1e6,
        "CI Low (M)"           : (FORECAST_2026 + gained_lo) / 1e6,
        "CI High (M)"          : (FORECAST_2026 + gained_hi) / 1e6,
        "Lift vs Forecast %"   : gained_pt / FORECAST_2026 * 100,
        "Confidence"           : r["Confidence"],
    })

proj_df = pd.DataFrame(proj_rows)

# Display table
from IPython.display import display as ipy_display
ipy_display(
    proj_df.style
    .format({
        "2026 Forecast (M)": "{:.2f}",
        "Scenario Lift (M)": "+{:.2f}",
        "Projected 2026 (M)": "{:.2f}",
        "CI Low (M)": "{:.2f}",
        "CI High (M)": "{:.2f}",
        "Lift vs Forecast %": "+{:.2f}%",
    })
    .set_caption("2026 Attendance Projections — Forecast (46.4M) + Media Scenario Lift")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "13px"), ("font-weight", "bold"),
                   ("color", "#002B5C"), ("text-align", "left")]},
        {"selector": "th",
         "props": [("background-color", "#002B5C"), ("color", "white"),
                   ("font-weight", "bold"), ("text-align", "center")]},
    ])
)

# ── Chart: 2026 projected attendance by scenario ──────────────────────────────
fig, ax = plt.subplots(figsize=(16, 7))
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.2, color=CPX_LGREY)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

x_pos  = np.arange(len(proj_df))
labels = proj_df["Scenario"].tolist()
colors = [bar_color(r) for _, r in proj_df.iterrows()]

projected = proj_df["Projected 2026 (M)"].values
ci_lo     = (proj_df["Projected 2026 (M)"] - proj_df["CI Low (M)"]).values
ci_hi     = (proj_df["CI High (M)"] - proj_df["Projected 2026 (M)"]).values

# Baseline reference line
ax.axhline(FORECAST_2026 / 1e6, color=CPX_RED, lw=1.5, ls="--", alpha=0.7,
           label=f"2026 Forecast: {FORECAST_2026/1e6:.1f}M")

# Bars showing projected total
ax.bar(x_pos, projected, color=colors, alpha=0.88, edgecolor="white", width=0.6)
ax.errorbar(x_pos, projected, yerr=[ci_lo, ci_hi],
            fmt="none", ecolor=CPX_DGREY, capsize=5, alpha=0.7, lw=1.5)

# Labels
for i, row in proj_df.iterrows():
    lift = row["Scenario Lift (M)"]
    ax.text(i, projected[i] + ci_hi[i] + 0.05,
            f"+{lift:.2f}M\n({row['Lift vs Forecast %']:+.2f}%)",
            ha="center", va="bottom", fontsize=8.5,
            fontweight="bold", color=CPX_NAVY)
    ax.text(i, projected[i] * 0.5,
            f"{projected[i]:.2f}M",
            ha="center", va="center", fontsize=9,
            fontweight="bold", color="white")

ax.set_xticks(x_pos)
ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=9, color=CPX_DGREY)
ax.set_ylabel("Projected 2026 Attendance (M)", fontsize=10, color=CPX_DGREY)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.1f}M"))
ax.set_ylim(45, max(projected + ci_hi) * 1.06)
ax.tick_params(axis="y", labelsize=9, labelcolor=CPX_DGREY)
ax.set_title(
    "2026 Projected Attendance by Scenario\n"
    f"Baseline Forecast: {FORECAST_2026/1e6:.1f}M  +  Media Scenario Lift",
    fontweight="bold", color=CPX_NAVY, fontsize=13, pad=10
)

leg = ax.legend(fontsize=10, loc="upper left", framealpha=0.95, edgecolor=CPX_LGREY)

plt.tight_layout()
plt.savefig(f"{DECOMP_PATH}FINAL_2026_scenario_projections.png",
            dpi=160, bbox_inches="tight", facecolor="white")
plt.show()
print(f"\n2026 Forecast: {FORECAST_2026:,.0f}")
print(f"Best scenario ({proj_df.iloc[-1]['Scenario']}): "
      f"{proj_df.iloc[-1]['Projected 2026 (M)']:.2f}M "
      f"(+{proj_df.iloc[-1]['Scenario Lift (M)']:.2f}M)")
print("\u2713 Saved FINAL_2026_scenario_projections.png")

In [0]:
# ── EXH Annual Attendance Decomposition Table ─────────────────────────────────
import pandas as pd
import numpy as np
import pickle
import arviz as az
from IPython.display import display as ipy_display

# ── Paths & model artifacts ─────────────────────────────────────────────
ARTIFACT_PATH = GROUPED_PATH  # v2 path from cell 3
DECOMP_PATH   = DECOMP_PATH   # v2 path from cell 3

target_col = model_meta["target_col"]

with open(f"{ARTIFACT_PATH}spend_train_max_iter2.pkl", "rb") as f:
    spend_train_max = pickle.load(f)
with open(f"{ARTIFACT_PATH}scaler_y.pkl", "rb") as f:
    _sy = pickle.load(f)

scale_factor = float(_sy.scale_[0])  # 337,563.61 (StandardScaler scale)

# Posterior mean gamma (= beta coefficient) per combo
trace = az.from_netcdf(f"{ARTIFACT_PATH}trace_iter2_mixed_hier.nc")
post  = trace.posterior
gamma_means = {combo: float(post[f"gamma_{combo}"].mean()) for combo in ACTIVE_COMBOS}

# ── Adstock & saturation functions ────────────────────────────────────────────
def adstock_geometric(x, alpha):
    out = np.zeros_like(x, dtype=float)
    out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = x[t] + alpha * out[t - 1]
    return out

def hill_sat(x, k, alpha_s):
    return (x ** alpha_s) / (x ** alpha_s + k ** alpha_s)

def logistic_sat(x, lam):
    return 1 - np.exp(-lam * x)

years_avail = [2023, 2024, 2025]

baseline_vals      = []
awareness_vals     = []
consideration_vals = []
conversion_vals    = []
actual_vals        = []

for yr in years_avail:
    mask = df_full["year"] == yr
    df_yr = df_full[mask].copy()

    actual_vals.append(df_yr[target_col].sum())

    # Baseline = total actual - sum of all media contributions
    media_contrib_total = 0.0
    aw_contrib = 0.0
    co_contrib = 0.0
    cv_contrib = 0.0

    for combo in ACTIVE_COMBOS:
        funnel = combo.split("__")[0]
        group  = combo.split("__")[1]
        p      = response_params[combo]
        alpha  = p["alpha"]
        sat    = p["saturation"]

        # Get spend cols for this combo's LOBs
        lobs_in_combo = LOB_GROUP_MAP.get(group, [])
        spend_cols = [
            c for c in df_full.columns
            if c.startswith(f"{funnel}_")
            and "__" not in c
            and not c.startswith("media_")
            and any(lob.lower() in c.lower() for lob in lobs_in_combo)
        ]
        if not spend_cols:
            continue

        spend_yr = df_yr[spend_cols].sum(axis=1).values

        # Apply adstock
        adstocked = adstock_geometric(spend_yr, alpha)

        # Normalize by training max
        normalized = adstocked / spend_train_max[combo]

        # Apply saturation
        if sat == "hill":
            transformed = hill_sat(normalized, p["k"], p["alpha_s"])
        else:
            transformed = logistic_sat(normalized, p["lam"])

        # Contribution = posterior_gamma * saturated_sum * scale_factor
        contrib = gamma_means[combo] * transformed.sum() * scale_factor

        if   funnel == "Awareness":     aw_contrib += contrib
        elif funnel == "Consideration": co_contrib += contrib
        elif funnel == "Conversion":    cv_contrib += contrib

    media_contrib_total = aw_contrib + co_contrib + cv_contrib
    baseline_vals.append(df_yr[target_col].sum() - media_contrib_total)
    awareness_vals.append(aw_contrib)
    consideration_vals.append(co_contrib)
    conversion_vals.append(cv_contrib)

# ── Build table ───────────────────────────────────────────────────────────────
rows = []
for i, yr in enumerate(years_avail):
    act  = actual_vals[i]
    base = baseline_vals[i]
    aw   = awareness_vals[i]
    co   = consideration_vals[i]
    cv   = conversion_vals[i]
    med  = aw + co + cv

    yr_label = str(yr)
    rows.append({
        "Metric" : "Baseline (non-media)",
        yr_label : f"{base/1e6:.1f}M ({base/act*100:.0f}%)"
    })
    rows.append({
        "Metric" : "Total Media Attributed",
        yr_label : f"{med/1e6:.1f}M ({med/act*100:.0f}%)"
    })

# Pivot to wide
table = pd.DataFrame([
    {
        "Metric"         : "Baseline (non-media)",
        "2023"           : f"{baseline_vals[0]/1e6:.1f}M ({baseline_vals[0]/actual_vals[0]*100:.0f}%)",
        "2024"           : f"{baseline_vals[1]/1e6:.1f}M ({baseline_vals[1]/actual_vals[1]*100:.0f}%)",
        "2025" : f"{baseline_vals[2]/1e6:.1f}M ({baseline_vals[2]/actual_vals[2]*100:.0f}%)",
    },
    {
        "Metric"         : "Total Media Attributed",
        "2023"           : f"{awareness_vals[0]+consideration_vals[0]+conversion_vals[0]:.1f}M ({(awareness_vals[0]+consideration_vals[0]+conversion_vals[0])/actual_vals[0]*100:.0f}%)",
        "2024"           : f"{awareness_vals[1]+consideration_vals[1]+conversion_vals[1]:.1f}M ({(awareness_vals[1]+consideration_vals[1]+conversion_vals[1])/actual_vals[1]*100:.0f}%)",
        "2025" : f"{awareness_vals[2]+consideration_vals[2]+conversion_vals[2]:.1f}M ({(awareness_vals[2]+consideration_vals[2]+conversion_vals[2])/actual_vals[2]*100:.0f}%)",
    },
]).set_index("Metric")

In [0]:
# ── Full decomposition table with funnel-level media breakdown ────────────────
from IPython.display import display as ipy_display

decomp_display = pd.DataFrame([
    {
        "Metric"  : "Total Actual Attendance",
        "2023"    : f"{actual_vals[0]/1e6:.2f}M",
        "2024"    : f"{actual_vals[1]/1e6:.2f}M",
        "2025"    : f"{actual_vals[2]/1e6:.2f}M",
    },
    {
        "Metric"  : "Baseline (Non-Media)",
        "2023"    : f"{baseline_vals[0]/1e6:.2f}M ({baseline_vals[0]/actual_vals[0]*100:.0f}%)",
        "2024"    : f"{baseline_vals[1]/1e6:.2f}M ({baseline_vals[1]/actual_vals[1]*100:.0f}%)",
        "2025"    : f"{baseline_vals[2]/1e6:.2f}M ({baseline_vals[2]/actual_vals[2]*100:.0f}%)",
    },
    {
        "Metric"  : "Total Media Contribution",
        "2023"    : f"{(awareness_vals[0]+consideration_vals[0]+conversion_vals[0])/1e6:.2f}M ({(awareness_vals[0]+consideration_vals[0]+conversion_vals[0])/actual_vals[0]*100:.0f}%)",
        "2024"    : f"{(awareness_vals[1]+consideration_vals[1]+conversion_vals[1])/1e6:.2f}M ({(awareness_vals[1]+consideration_vals[1]+conversion_vals[1])/actual_vals[1]*100:.0f}%)",
        "2025"    : f"{(awareness_vals[2]+consideration_vals[2]+conversion_vals[2])/1e6:.2f}M ({(awareness_vals[2]+consideration_vals[2]+conversion_vals[2])/actual_vals[2]*100:.0f}%)",
    },
    {
        "Metric"  : "  └ Awareness",
        "2023"    : f"{awareness_vals[0]/1e6:.3f}M ({awareness_vals[0]/actual_vals[0]*100:.1f}%)",
        "2024"    : f"{awareness_vals[1]/1e6:.3f}M ({awareness_vals[1]/actual_vals[1]*100:.1f}%)",
        "2025"    : f"{awareness_vals[2]/1e6:.3f}M ({awareness_vals[2]/actual_vals[2]*100:.1f}%)",
    },
    {
        "Metric"  : "  └ Consideration",
        "2023"    : f"{consideration_vals[0]/1e6:.3f}M ({consideration_vals[0]/actual_vals[0]*100:.1f}%)",
        "2024"    : f"{consideration_vals[1]/1e6:.3f}M ({consideration_vals[1]/actual_vals[1]*100:.1f}%)",
        "2025"    : f"{consideration_vals[2]/1e6:.3f}M ({consideration_vals[2]/actual_vals[2]*100:.1f}%)",
    },
    {
        "Metric"  : "  └ Conversion",
        "2023"    : f"{conversion_vals[0]/1e6:.3f}M ({conversion_vals[0]/actual_vals[0]*100:.1f}%)",
        "2024"    : f"{conversion_vals[1]/1e6:.3f}M ({conversion_vals[1]/actual_vals[1]*100:.1f}%)",
        "2025"    : f"{conversion_vals[2]/1e6:.3f}M ({conversion_vals[2]/actual_vals[2]*100:.1f}%)",
    },
    {
        "Metric"  : "Media Share %",
        "2023"    : f"{(awareness_vals[0]+consideration_vals[0]+conversion_vals[0])/actual_vals[0]*100:.1f}%",
        "2024"    : f"{(awareness_vals[1]+consideration_vals[1]+conversion_vals[1])/actual_vals[1]*100:.1f}%",
        "2025"    : f"{(awareness_vals[2]+consideration_vals[2]+conversion_vals[2])/actual_vals[2]*100:.1f}%",
    },
]).set_index("Metric")

ipy_display(
    decomp_display.style
    .set_caption("EXH Annual Attendance Decomposition (Full Year, 52 wks)")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "13px"), ("font-weight", "bold"),
                   ("color", "#002B5C"), ("text-align", "left")]},
        {"selector": "th",
         "props": [("background-color", "#002B5C"), ("color", "white"),
                   ("font-weight", "bold"), ("text-align", "center")]},
        {"selector": "td",
         "props": [("min-width", "140px")]},
    ])
)

print("\n" + decomp_display.to_string())
print(f"\n2025 coverage: full 52-week year (Jan 6 – Dec 29)")

In [0]:
# ── EXH Annual Attendance Decomposition Table — Standalone ───────────────────
# Loads from saved artifacts. Drop anywhere in Budget Scenarios notebook.

import pickle
import shutil
import pandas as pd
import numpy as np
from IPython.display import display

# ── Paths ─────────────────────────────────────────────────────────────────────
ARTIFACT_PATH = GROUPED_PATH  # v2 path from cell 3
DECOMP_PATH   = DECOMP_PATH   # v2 path from cell 3
OUT_PATH      = DECOMP_PATH   # v2 path from cell 3

# ── Load decomp artifact ──────────────────────────────────────────────────────
with open(f"{DECOMP_PATH}lob_decomposition.pkl", "rb") as f:
    _da = pickle.load(f)
decomp = _da["decomp_table"]        # LOB-level graft decomp

# ── Load model artifacts ──────────────────────────────────────────────────────
df_full = pd.read_parquet(f"{ARTIFACT_PATH}df_full.parquet")
df_full["year"] = df_full["week_date"].dt.year

with open(f"{ARTIFACT_PATH}model_meta.pkl", "rb") as f:
    model_meta = pickle.load(f)
target_col   = model_meta["target_col"]
ACTIVE_COMBOS = model_meta["ACTIVE_COMBOS"]
LOB_GROUP_MAP = model_meta["LOB_GROUP_MAP"]

with open(f"{ARTIFACT_PATH}response_params.pkl", "rb") as f:
    response_params = pickle.load(f)

with open(f"{ARTIFACT_PATH}spend_train_max_iter2.pkl", "rb") as f:
    spend_train_max = pickle.load(f)
with open(f"{ARTIFACT_PATH}scaler_y.pkl", "rb") as f:
    _sy = pickle.load(f)
scale_factor = float(_sy.scale_[0])  # 337,563.61 — converts model units → attendance

# ── Adstock & saturation helpers ─────────────────────────────────────────────
def _adstock(x, alpha):
    out = np.zeros_like(x, dtype=float)
    out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = x[t] + alpha * out[t - 1]
    return out

def _logistic(x, lam):
    return 1 - np.exp(-lam * x)

def _hill(x, k, alpha_s):
    return (x ** alpha_s) / (x ** alpha_s + k ** alpha_s)

# ── Compute per-year funnel contributions ────────────────────────────────────
years_avail   = [2023, 2024, 2025]
awareness_vals, consideration_vals, conversion_vals = [], [], []
actual_vals_sa = []

for yr in years_avail:
    df_yr = df_full[df_full["year"] == yr].copy()
    actual_vals_sa.append(df_yr[target_col].sum())
    aw, co, cv = 0.0, 0.0, 0.0
    for combo in ACTIVE_COMBOS:
        funnel = combo.split("__")[0]
        group  = combo.split("__")[1]
        p      = response_params[combo]
        lobs   = LOB_GROUP_MAP.get(group, [])
        cols   = [
            c for c in df_full.columns
            if c.startswith(f"{funnel}_") and "__" not in c
            and not c.startswith("media_")
            and any(lob.lower() in c.lower() for lob in lobs)
        ]
        if not cols:
            continue
        # Normalize spend to [0,1] BEFORE adstocking — matches combo_contribution()
        norm_spend = df_yr[cols].sum(axis=1).values / spend_train_max[combo]
        adstocked   = _adstock(norm_spend, p["alpha"])
        transformed = _hill(adstocked, p["k"], p["alpha_s"]) if p["saturation"] == "hill" else _logistic(adstocked, p["lam"])
        contrib     = p["gamma"] * transformed.sum() * scale_factor
        if   funnel == "Awareness":     aw += contrib
        elif funnel == "Consideration": co += contrib
        elif funnel == "Conversion":    cv += contrib
    awareness_vals.append(aw)
    consideration_vals.append(co)
    conversion_vals.append(cv)

actual_by_year = dict(zip(years_avail, actual_vals_sa))

# ── Build funnel-by-year lookup ───────────────────────────────────────────────
funnel_by_year = pd.DataFrame(
    {
        "Awareness":     awareness_vals,
        "Consideration": consideration_vals,
        "Conversion":    conversion_vals,
    },
    index=years_avail,
)

# ── Build summary table ───────────────────────────────────────────────────────
rows = []
for yr in years_avail:
    act  = actual_by_year[yr]
    aw   = funnel_by_year.loc[yr, "Awareness"]     if "Awareness"     in funnel_by_year.columns else 0
    co   = funnel_by_year.loc[yr, "Consideration"] if "Consideration" in funnel_by_year.columns else 0
    cv   = funnel_by_year.loc[yr, "Conversion"]    if "Conversion"    in funnel_by_year.columns else 0
    med  = aw + co + cv
    base = act - med
    yr_label = str(yr)

    rows.append({"Metric": "Baseline (non-media)",   "Year": yr_label, "Value_M": base/1e6, "Pct": base/act})
    rows.append({"Metric": "Total Media Attributed", "Year": yr_label, "Value_M": med/1e6,  "Pct": med/act})
    rows.append({"Metric": "— Awareness",            "Year": yr_label, "Value_M": aw/1e6,   "Pct": aw/act})
    rows.append({"Metric": "— Consideration",        "Year": yr_label, "Value_M": co/1e6,   "Pct": co/act})
    rows.append({"Metric": "— Conversion",           "Year": yr_label, "Value_M": cv/1e6,   "Pct": cv/act})

df_rows = pd.DataFrame(rows)
df_rows["Display"] = (
    df_rows["Value_M"].apply(lambda x: f"{x:.1f}M") + 
    " (" + 
    df_rows["Pct"].apply(lambda x: f"{x*100:.0f}%") + 
    ")"
)

table = (
    df_rows
    .pivot(index="Metric", columns="Year", values="Display")
    .reindex(
        ["Baseline (non-media)", "Total Media Attributed",
         "— Awareness", "— Consideration", "— Conversion"],
        axis=0
    )
    .reindex(["2023", "2024", "2025"], axis=1)
)

display(
    table.style
    .set_caption("EXH Annual Attendance Decomposition — Baseline vs Media Attributed")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "13px"), ("font-weight", "bold"),
                   ("color", "#002B5C"), ("text-align", "left")]},
        {"selector": "th",
         "props": [("background-color", "#002B5C"), ("color", "white"),
                   ("font-weight", "bold"), ("text-align", "center")]},
        {"selector": "tr:nth-child(2)",
         "props": [("font-weight", "bold"), ("background-color", "#F2F2F2")]},
    ])
)
print("✓ Decomposition table rendered.")

In [0]:
table.display()

## 2025 Attendance Structure
Where the 42.51M comes from, and the contested geo-test attribution shown inside (not on top).

In [0]:
fig, ax = plt.subplots(figsize=(12, 3.2))
bar_h = 0.46

# ── Stacked bars ──────────────────────────────────────────────────────────────
ax.barh([0], [controls_base], height=bar_h, color=CPX_GREY, edgecolor="white", linewidth=2)
ax.barh([0], [media_base], left=[controls_base], height=bar_h,
        color=CPX_NAVY, edgecolor="white", linewidth=2)

# ── Inline labels (replace external legend) ───────────────────────────────────
ax.text(controls_base * 0.5, 0,
        f"Baseline / Non-Media\n{controls_base/1e6:.1f}M   ·   {controls_base/actual_total*100:.0f}%",
        ha="center", va="center", fontsize=10.5, fontweight="bold", color="white")
ax.text(controls_base + media_base * 0.5, 0,
        f"Validated Media\n{media_base/1e6:.1f}M   ·   {MEDIA_SHARE*100:.0f}%",
        ha="center", va="center", fontsize=10.5, fontweight="bold", color="white")

# ── Total callout on right edge ────────────────────────────────────────────────
ax.text(actual_total * 1.012, 0, f"= {actual_total:,.0f}",
        ha="left", va="center", fontsize=9.5, fontweight="bold", color=CPX_NAVY)

# ── Contested — clean note below bar (no distracting arrow) ───────────────────
ax.vlines(actual_total, -bar_h / 2, -bar_h / 2 - 0.07, color=CPX_RED, lw=1.1)
ax.text(actual_total, -bar_h / 2 - 0.10,
        f"⚠  {geo_contested:,.0f} contested attribution (geo-test LOBs — hypothesised inside total, not additive, unproven)",
        ha="right", va="top", fontsize=8, color=CPX_RED, style="italic")

# ── Clean axes ────────────────────────────────────────────────────────────────
ax.set_xlim(0, actual_total * 1.20)
ax.set_ylim(-0.52, 0.36)
ax.set_yticks([])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1e6:.0f}M"))
ax.tick_params(axis="x", labelsize=8.5, labelcolor="#777777")
ax.spines["bottom"].set_color("#DDDDDD")
for sp in ["top", "right", "left"]:
    ax.spines[sp].set_visible(False)

ax.set_title(f"2025 Total Attendance = {actual_total:,.0f}",
             fontweight="bold", color=CPX_NAVY, fontsize=13, pad=10, loc="left")

plt.tight_layout()
plt.savefig(f"{DECOMP_PATH}FINAL_attendance_structure.png",
            dpi=160, bbox_inches="tight", facecolor="white")
plt.show()
print("✓ Saved FINAL_attendance_structure.png")

In [0]:
# ── CELL: LOB Spend × Scenario Summary Table ─────────────────────────────────
# Rows = all original LOBs from funnel_lob_cols (not just decomp-validated).
# Columns = Actual 2025 Spend, then per scenario:
#   Scenario Spend | Δ Spend | Expected Attendance Lift (with 89% CI)
# Geo-test LOBs flagged in table and Excel.

import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import shutil
import os

# ── 1. Scenario list ──────────────────────────────────────────────────────────
scenarios = (
    [(r["label"],
      {c: baseline_spend[c] * (1 + r["spend_increase_pct"] / 100) for c in ACTIVE_COMBOS},
      r)
     for r in s1_results]
    + [("Reshuffle (10% cap)", s2_alloc, r2)]
    + [(r["label"],
        {c: reshuffle({c2: baseline_spend[c2] * (1 + r["spend_increase_pct"] / 100)
                       for c2 in ACTIVE_COMBOS})[c]
         for c in ACTIVE_COMBOS},
        r)
       for r in s3_results]
)

# ── 2. LOB-level att-per-$1 from decomp ──────────────────────────────────────
fl_rate_map = {}
for _, row in decomp_valid.iterrows():
    key = (row["Funnel"], row["LOB"])
    if pd.notna(row["Att per $1"]) and row["Annual Spend"] > 0:
        fl_rate_map.setdefault(key, []).append(
            (float(row["Att per $1"]), float(row["Annual Spend"]))
        )
fl_rate = {}
for key, vals in fl_rate_map.items():
    total_sp = sum(v[1] for v in vals)
    fl_rate[key] = sum(v[0] * v[1] for v in vals) / total_sp if total_sp > 0 else 0.0

# ── 3. Geo-test flags ─────────────────────────────────────────────────────────
geo_test_lobs = set(decomp_artifact["geo_test_headline"])

# ── Load funnel_lob_cols and helpers if not in scope ─────────────────────────
if "funnel_lob_cols" not in dir():
    from collections import defaultdict

    df_full = pd.read_parquet(f"{GROUPED_PATH}df_full.parquet")

    media_cols = [c for c in df_full.columns if any(
        c.startswith(f"{f}_") for f in ["Awareness", "Consideration", "Conversion"]
    ) and "__" not in c and "media_" not in c]

    funnel_lob_cols = defaultdict(list)
    for col in media_cols:
        parts = col.split("_", 1); funnel = parts[0]; rest = parts[1]
        funnel_lob_cols[(funnel, rest[:rest.rfind("_")])].append(col)

    print(f"Built funnel_lob_cols: {len(funnel_lob_cols)} (funnel, LOB) cells")

if "lob_display_name" not in dir():
    def lob_display_name(raw):
        return {
            "EXH__Brand": "EXH - Brand", "FandB": "F&B", "GandE": "G&E",
            "Film_Marketing__Inform": "Film Marketing - Inform",
            "Performance__Convert_and_Upgrade": "Performance - Convert & Upgrade",
            "Premium_Experiences": "Premium Experiences - AVX",
            "Premium_Experiences__AVX": "Premium Experiences - AVX",
            "Premium_Experiences__IMAX": "Premium Experiences - IMAX",
            "Premium_Experiences__VIP": "Premium Experiences - VIP",
            "Prepaid__B2B": "Prepaid - B2B", "Prepaid__B2C": "Prepaid - B2C",
            "Cineclub": "Cineclub",
        }.get(raw, raw)

if "get_parent_combo" not in dir():
    def get_parent_combo(funnel, lob_raw):
        for group_name, lob_list in LOB_GROUP_MAP.items():
            if lob_raw in lob_list:
                combo = f"{funnel}__{group_name}"
                if combo in ACTIVE_COMBOS:
                    return combo, group_name, "validated"
                return None, group_name, "dropped"
        return None, None, "no_group"

# ── 4. Build rows — all original LOBs from funnel_lob_cols ───────────────────
rows = []
for (funnel, lob_raw), cols in sorted(funnel_lob_cols.items()):
    disp    = lob_display_name(lob_raw)
    base_sp = float(df_full[df_full["year"] == 2025][cols].sum().sum())
    if base_sp <= 0:
        continue

    rate   = fl_rate.get((funnel, disp), 0.0)
    is_geo = disp in geo_test_lobs

    row_data = {
        "Funnel"            : funnel,
        "LOB"               : disp,
        "Geo Test"          : "⚠ Geo Test" if is_geo else "",
        "Actual Spend 2025" : round(base_sp, 0),
    }

    for scen_label, group_alloc, lift_r in scenarios:
        combo, _, routing = get_parent_combo(funnel, lob_raw)
        if routing == "validated" and combo in group_alloc:
            scale = group_alloc[combo] / baseline_spend[combo] if baseline_spend[combo] > 0 else 1.0
        else:
            scale = 1.0  # geo-test / dropped — held fixed

        scen_sp       = base_sp * scale
        delta_sp      = scen_sp - base_sp
        lob_att_delta = delta_sp * rate

        total_delta = sum(
            (group_alloc.get(c, 0.0) - baseline_spend.get(c, 0.0))
            for c in ACTIVE_COMBOS
        )
        lob_share_of_delta = (delta_sp / total_delta) if abs(total_delta) > 1 else 0.0
        ci_lo = lift_r["pct_lo"] * lob_share_of_delta
        ci_hi = lift_r["pct_hi"] * lob_share_of_delta

        safe = scen_label.replace(" ", "_").replace("%", "pct").replace("+", "plus")
        row_data[f"{safe}__spend"]    = round(scen_sp, 0)
        row_data[f"{safe}__delta"]    = round(delta_sp, 0)
        row_data[f"{safe}__att_lift"] = round(lob_att_delta, 0)
        row_data[f"{safe}__ci"]       = f"[{ci_lo:+.1f}%, {ci_hi:+.1f}%]"

    rows.append(row_data)

summary_df = pd.DataFrame(rows)
display(summary_df)

# ── 5. Excel export — CPX styled ─────────────────────────────────────────────
NAVY  = "002B5C"; WHITE = "FFFFFF"; AMBER = "FFF2CC"
SCEN_FILLS = {
    "plus10pct_budget"         : "DDEEFF",
    "plus20pct_budget"         : "BBDDFF",
    "plus30pct_budget"         : "88BBFF",
    "Reshuffle__10pct_cap_"    : "FFF2CC",
    "plus10pct_plus_reshuffle" : "DDFFDD",
    "plus20pct_plus_reshuffle" : "BBFFBB",
    "plus30pct_plus_reshuffle" : "88EE88",
}
FUNNEL_ROW_FILLS = {
    "Awareness"    : "EEF3FA",
    "Consideration": "FDEAEA",
    "Conversion"   : "FFFBEA",
}

wb = Workbook()
ws = wb.active
ws.title = "LOB Scenario Spend"

hdr_font  = Font(name="Arial", bold=True, color=WHITE, size=10)
hdr_fill  = PatternFill("solid", fgColor=NAVY)
sub_font  = Font(name="Arial", bold=True, color=NAVY, size=9)
cell_font = Font(name="Arial", size=9)
geo_font  = Font(name="Arial", bold=True, size=9, color="C05000")
geo_fill  = PatternFill("solid", fgColor=AMBER)
center    = Alignment(horizontal="center", vertical="center", wrap_text=True)
left      = Alignment(horizontal="left",   vertical="center")
thin      = Side(style="thin", color="D0D0D0")
border    = Border(thin, thin, thin, thin)

# Title
ws["A1"] = "EXH MMM — LOB Spend by Scenario (2025, original LOBs, graft-split)"
ws["A1"].font = Font(name="Arial", bold=True, size=12, color=NAVY)
ws.merge_cells("A1:Z1")
ws.append([])  # row 2 blank

# Scenario column groups
fixed_cols = ["Funnel", "LOB", "Geo Test", "Actual Spend 2025 ($)"]
scen_col_groups = []
for scen_label, _, lift_r in scenarios:
    safe = scen_label.replace(" ", "_").replace("%", "pct").replace("+", "plus")
    scen_col_groups.append({
        "label"  : scen_label,
        "safe"   : safe,
        "lift_r" : lift_r,
        "subcols": [
            "Scenario Spend ($)",
            "Δ vs Baseline ($)",
            "LOB Att Lift (att)",
            "89% CI (model %)",
        ]
    })

# Row 3 — top headers (fixed cols span rows 3-4; scenario cols span subcols)
r3 = 3
for j, h in enumerate(fixed_cols, 1):
    c = ws.cell(row=r3, column=j, value=h)
    c.font = hdr_font; c.fill = hdr_fill; c.alignment = center; c.border = border
    ws.merge_cells(start_row=r3, start_column=j, end_row=r3+1, end_column=j)

col_cursor = len(fixed_cols) + 1
for grp in scen_col_groups:
    merge_end = col_cursor + len(grp["subcols"]) - 1
    lr = grp["lift_r"]
    header_val = (
        f"{grp['label']}  |  Lift: +{lr['lift_pct']:.1f}%  "
        f"CI: [+{lr['pct_lo']:.1f}%, +{lr['pct_hi']:.1f}%]  "
        f"Conf: {lr['confidence']}"
    )
    fill_key  = next((k for k in SCEN_FILLS if k in grp["safe"]), None)
    scen_fill = PatternFill("solid", fgColor=SCEN_FILLS.get(fill_key, "EEEEEE"))
    c = ws.cell(row=r3, column=col_cursor, value=header_val)
    c.font = Font(name="Arial", bold=True, color=NAVY, size=9)
    c.fill = scen_fill; c.alignment = center; c.border = border
    ws.merge_cells(start_row=r3, start_column=col_cursor,
                   end_row=r3,   end_column=merge_end)
    col_cursor = merge_end + 1

# Row 4 — sub-column headers
r4 = 4
col_cursor = len(fixed_cols) + 1
for grp in scen_col_groups:
    fill_key  = next((k for k in SCEN_FILLS if k in grp["safe"]), None)
    scen_fill = PatternFill("solid", fgColor=SCEN_FILLS.get(fill_key, "EEEEEE"))
    for sub in grp["subcols"]:
        c = ws.cell(row=r4, column=col_cursor, value=sub)
        c.font = sub_font; c.fill = scen_fill
        c.alignment = center; c.border = border
        col_cursor += 1

# Data rows
for row_data in rows:
    funnel   = row_data["Funnel"]
    is_geo   = row_data["Geo Test"] != ""
    row_fill = PatternFill("solid", fgColor=FUNNEL_ROW_FILLS.get(funnel, "FFFFFF"))

    data_row = [
        row_data["Funnel"],
        row_data["LOB"],
        row_data["Geo Test"],
        row_data["Actual Spend 2025"],
    ]
    for grp in scen_col_groups:
        s = grp["safe"]
        data_row += [
            row_data.get(f"{s}__spend",    ""),
            row_data.get(f"{s}__delta",    ""),
            row_data.get(f"{s}__att_lift", ""),
            row_data.get(f"{s}__ci",       ""),
        ]

    ws.append(data_row)
    dr = ws.max_row
    for j in range(1, len(data_row) + 1):
        cc = ws.cell(row=dr, column=j)
        cc.border    = border
        cc.alignment = left if j <= 2 else center
        if j == 3 and is_geo:
            cc.font = geo_font
            cc.fill = geo_fill
        elif is_geo:
            cc.font = Font(name="Arial", size=9, italic=True, color="885000")
            cc.fill = PatternFill("solid", fgColor="FFFAF0")
        else:
            cc.font = cell_font
            cc.fill = row_fill

# Column widths
ws.column_dimensions["A"].width = 16
ws.column_dimensions["B"].width = 30
ws.column_dimensions["C"].width = 14
ws.column_dimensions["D"].width = 18
col_cursor = 5
for grp in scen_col_groups:
    for w in [16, 16, 16, 20]:
        ws.column_dimensions[get_column_letter(col_cursor)].width = w
        col_cursor += 1

ws.freeze_panes         = "D5"
ws.row_dimensions[3].height = 40
ws.row_dimensions[4].height = 28

_tmp      = "/tmp/I2_B10_LOB_Scenario_Spend_Table.xlsx"
xlsx_path = os.path.join(OUT_PATH, "I2_B10_LOB_Scenario_Spend_Table.xlsx")
wb.save(_tmp)
shutil.copy2(_tmp, xlsx_path)
os.remove(_tmp)
print(f"✓ Saved: {xlsx_path}")

In [0]:
# ── CELL: LOB Spend × Scenario Summary Table ─────────────────────────────────
# Rows = (Funnel, LOB). Columns = Actual 2025 Spend, then per scenario:
#   Scenario Spend | Δ Spend | Expected Attendance Lift (with 89% CI)

import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
import os

# ── 1. Scenario list (same as chart cell) ────────────────────────────────────
scenarios = (
    [(r["label"],
      {c: baseline_spend[c] * (1 + r["spend_increase_pct"] / 100) for c in ACTIVE_COMBOS},
      r)
     for r in s1_results]
    + [("Reshuffle (10% cap)", s2_alloc, r2)]
    + [(r["label"],
        {c: reshuffle({c2: baseline_spend[c2] * (1 + r["spend_increase_pct"] / 100)
                       for c2 in ACTIVE_COMBOS})[c]
         for c in ACTIVE_COMBOS},
        r)
       for r in s3_results]
)

# ── 2. LOB-level att-per-$1 from decomp (for LOB-level contribution lift) ────
# Scenario attendance lift is model-level (group curves), but we can attribute
# the spend delta × att-per-$1 per LOB as the directional LOB contribution.
fl_rate_map = {}
for _, row in decomp_valid.iterrows():
    key = (row["Funnel"], row["LOB"])
    if pd.notna(row["Att per $1"]) and row["Annual Spend"] > 0:
        fl_rate_map.setdefault(key, []).append(
            (float(row["Att per $1"]), float(row["Annual Spend"]))
        )
# Spend-weighted average per (funnel, lob)
fl_rate = {}
for key, vals in fl_rate_map.items():
    total_sp = sum(v[1] for v in vals)
    fl_rate[key] = sum(v[0] * v[1] for v in vals) / total_sp if total_sp > 0 else 0.0

# ── 3. Build rows ─────────────────────────────────────────────────────────────
rows = []
for funnel, lob in lob_keys:
    base_sp  = baseline_lob_spend.get((funnel, lob), 0.0)
    rate     = fl_rate.get((funnel, lob), 0.0)
    row_data = {
        "Funnel"        : funnel,
        "LOB"           : lob,
        "Actual Spend 2025": round(base_sp, 0),
    }
    for scen_label, group_alloc, lift_r in scenarios:
        scen_lob_spend = split_to_lobs(group_alloc)
        scen_sp  = scen_lob_spend.get((funnel, lob), 0.0)
        delta_sp = scen_sp - base_sp

        # LOB-level att contribution delta (directional, linear approximation)
        lob_att_delta = delta_sp * rate

        # Model-level CI scaled by this LOB's share of total scenario spend delta
        total_delta = sum(
            split_to_lobs(group_alloc).get(k, 0.0) - baseline_lob_spend.get(k, 0.0)
            for k in lob_keys
        )
        lob_share_of_delta = (delta_sp / total_delta) if abs(total_delta) > 1 else 0.0
        ci_lo = lift_r["pct_lo"] * lob_share_of_delta
        ci_hi = lift_r["pct_hi"] * lob_share_of_delta

        safe = scen_label.replace(" ", "_").replace("%", "pct").replace("+", "plus")
        row_data[f"{safe}__spend"]     = round(scen_sp, 0)
        row_data[f"{safe}__delta"]     = round(delta_sp, 0)
        row_data[f"{safe}__att_lift"]  = round(lob_att_delta, 0)
        row_data[f"{safe}__ci"]        = f"[{ci_lo:+.1f}%, {ci_hi:+.1f}%]"

    rows.append(row_data)

summary_df = pd.DataFrame(rows)

# ── 4. Display ────────────────────────────────────────────────────────────────
display(summary_df)

# ── 5. Excel export — CPX styled ─────────────────────────────────────────────
NAVY  = "002B5C"; WHITE = "FFFFFF"; LGREY = "F2F2F2"; AMBER = "FFF2CC"
SCEN_FILLS = {
    "+10pct_budget":       "DDEEFF",
    "+20pct_budget":       "BBDDFF",
    "+30pct_budget":       "88BBFF",
    "Reshuffle_(10pct_cap)": "FFF2CC",
    "plus10pct_plus_reshuffle": "DDFFDD",
    "plus20pct_plus_reshuffle": "BBFFBB",
    "plus30pct_plus_reshuffle": "88EE88",
}

wb  = Workbook()
ws  = wb.active
ws.title = "LOB Scenario Spend"

hdr_font  = Font(name="Arial", bold=True, color=WHITE, size=10)
hdr_fill  = PatternFill("solid", fgColor=NAVY)
sub_font  = Font(name="Arial", bold=True, color=NAVY, size=9)
cell_font = Font(name="Arial", size=9)
center    = Alignment(horizontal="center", vertical="center", wrap_text=True)
left      = Alignment(horizontal="left",   vertical="center")
thin      = Side(style="thin", color="D0D0D0")
border    = Border(thin, thin, thin, thin)

# Title
ws["A1"] = "EXH MMM — LOB Spend by Scenario (2025, graft-split)"
ws["A1"].font = Font(name="Arial", bold=True, size=12, color=NAVY)
ws.merge_cells("A1:Z1")

# Build header rows — Row 3: scenario group spans; Row 4: sub-columns
ws.append([])  # row 2 blank
fixed_cols = ["Funnel", "LOB", "Actual Spend 2025 ($)"]

# Collect scenario column groups
scen_col_groups = []
for scen_label, _, lift_r in scenarios:
    safe = scen_label.replace(" ", "_").replace("%", "pct").replace("+", "plus")
    scen_col_groups.append({
        "label"  : scen_label,
        "safe"   : safe,
        "lift_r" : lift_r,
        "subcols": [
            f"Scenario Spend ($)",
            f"Δ vs Baseline ($)",
            f"LOB Att Lift (att)",
            f"89% CI (model %)",
        ]
    })

# Row 3 — top headers
# ── 3. Build rows — all original LOBs from funnel_lob_cols ───────────────────
# Use fl_yr_spend for 2025 actual spend; flag geo-test LOBs from decomp artifact
geo_test_lobs = set(decomp_artifact["geo_test_headline"])  # from lob_decomposition.pkl

rows = []
for (funnel, lob_raw), cols in sorted(funnel_lob_cols.items()):
    disp     = lob_display_name(lob_raw)
    base_sp  = float(df_full[df_full["year"] == 2025][cols].sum().sum())
    if base_sp <= 0:
        continue

    rate     = fl_rate.get((funnel, disp), 0.0)
    is_geo   = disp in geo_test_lobs

    row_data = {
        "Funnel"           : funnel,
        "LOB"              : disp,
        "Geo Test"         : "⚠ Geo Test" if is_geo else "",
        "Actual Spend 2025": round(base_sp, 0),
    }

    for scen_label, group_alloc, lift_r in scenarios:
        # Scenario spend: scale the same cols from df_full by the group's spend ratio
        # Find parent combo for this (funnel, lob_raw)
        combo, _, routing = get_parent_combo(funnel, lob_raw)
        if routing == "validated" and combo in group_alloc:
            scale = group_alloc[combo] / baseline_spend[combo] if baseline_spend[combo] > 0 else 1.0
        else:
            scale = 1.0   # dropped/geo-test LOBs — spend held fixed

        scen_sp  = base_sp * scale
        delta_sp = scen_sp - base_sp
        lob_att_delta = delta_sp * rate

        total_delta = sum(
            (group_alloc.get(c, 0.0) - baseline_spend.get(c, 0.0))
            for c in ACTIVE_COMBOS
        )
        lob_share_of_delta = (delta_sp / total_delta) if abs(total_delta) > 1 else 0.0
        ci_lo = lift_r["pct_lo"] * lob_share_of_delta
        ci_hi = lift_r["pct_hi"] * lob_share_of_delta

        safe = scen_label.replace(" ", "_").replace("%", "pct").replace("+", "plus")
        row_data[f"{safe}__spend"]    = round(scen_sp, 0)
        row_data[f"{safe}__delta"]    = round(delta_sp, 0)
        row_data[f"{safe}__att_lift"] = round(lob_att_delta, 0)
        row_data[f"{safe}__ci"]       = f"[{ci_lo:+.1f}%, {ci_hi:+.1f}%]"

    rows.append(row_data)

summary_df = pd.DataFrame(rows)
display(summary_df)

# Row 4 — sub-column headers
r4 = 4
col_cursor = len(fixed_cols) + 1
for grp in scen_col_groups:
    fill_key  = next((k for k in SCEN_FILLS if k in grp["safe"]), None)
    scen_fill = PatternFill("solid", fgColor=SCEN_FILLS.get(fill_key, "EEEEEE"))
    for sub in grp["subcols"]:
        c = ws.cell(row=r4, column=col_cursor, value=sub)
        c.font = sub_font; c.fill = scen_fill
        c.alignment = center; c.border = border
        col_cursor += 1

# Data rows
FUNNEL_ROW_FILLS = {
    "Awareness"   : "EEF3FA",
    "Consideration": "FDEAEA",
    "Conversion"  : "FFFBEA",
}
for row_data in rows:
    funnel = row_data["Funnel"]
    row_fill = PatternFill("solid", fgColor=FUNNEL_ROW_FILLS.get(funnel, "FFFFFF"))
    data_row = [
        row_data["Funnel"],
        row_data["LOB"],
        row_data["Actual Spend 2025"],
    ]
    for grp in scen_col_groups:
        s = grp["safe"]
        data_row += [
            row_data.get(f"{s}__spend",    ""),
            row_data.get(f"{s}__delta",    ""),
            row_data.get(f"{s}__att_lift", ""),
            row_data.get(f"{s}__ci",       ""),
        ]
    ws.append(data_row)
    dr = ws.max_row
    for j in range(1, len(data_row) + 1):
        cc = ws.cell(row=dr, column=j)
        cc.font      = cell_font
        cc.fill      = row_fill
        cc.border    = border
        cc.alignment = left if j <= 2 else center

# Column widths
ws.column_dimensions["A"].width = 14
ws.column_dimensions["B"].width = 28
ws.column_dimensions["C"].width = 18
col_cursor = 4
for grp in scen_col_groups:
    for w in [16, 16, 16, 20]:
        from openpyxl.utils import get_column_letter
        ws.column_dimensions[get_column_letter(col_cursor)].width = w



        col_cursor += 1

ws.freeze_panes = "C5"
ws.row_dimensions[3].height = 36
ws.row_dimensions[4].height = 28

import shutil
_tmp = "/tmp/I2_B10_LOB_Scenario_Spend_Table.xlsx"
xlsx_path = os.path.join(OUT_PATH, "I2_B10_LOB_Scenario_Spend_Table.xlsx")
wb.save(_tmp)
shutil.copy2(_tmp, xlsx_path)
os.remove(_tmp)
print(f"✓ Saved: {xlsx_path}")

## Export — Formatted Excel + CSV
Two-sheet workbook: **Scenario Outcomes** (both frames, labeled, honest confidence) and
**Attendance Structure**. CSV backups alongside. Headers state units; the reshuffle row carries its
directional caveat.

In [0]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

NAVY="002B5C"; GREY="F2F2F2"; AMBER="FFF2CC"; WHITE="FFFFFF"
hdr_font = Font(name="Arial", bold=True, color=WHITE, size=10)
hdr_fill = PatternFill("solid", fgColor=NAVY)
cell_font = Font(name="Arial", size=10)
caveat_fill = PatternFill("solid", fgColor=AMBER)
center = Alignment(horizontal="center", vertical="center", wrap_text=True)
left   = Alignment(horizontal="left", vertical="center")
thin = Side(style="thin", color="D0D0D0"); border = Border(thin,thin,thin,thin)

wb = Workbook()

# ── Sheet 1: Scenario Outcomes ───────────────────────────────────────────────
ws = wb.active; ws.title = "Scenario Outcomes"
ws["A1"] = "EXH MMM — 2025 Budget Scenario Outcomes (reconciled to actual total attendance)"
ws["A1"].font = Font(name="Arial", bold=True, size=12, color=NAVY); ws.merge_cells("A1:I1")
ws["A2"] = (f"Real 2025 total attendance: {actual_total:,.0f}  |  media = {MEDIA_SHARE*100:.1f}% of total  "
            f"|  lifts scale validated media only  |  CI = 89% credible interval (model-conditional)")
ws["A2"].font = Font(name="Arial", italic=True, size=9, color="555555"); ws.merge_cells("A2:I2")

headers = ["Scenario","Media Lift %","Media Lift 89% CI","Admissions Gained",
           "Admissions Gained 89% CI","New Total Attendance","Total Lift %","Total Lift 89% CI","Confidence"]
ws.append([]); hr = 4
for j,h in enumerate(headers,1):
    c = ws.cell(row=hr, column=j, value=h); c.font=hdr_font; c.fill=hdr_fill; c.alignment=center; c.border=border

for _, r in final_df.iterrows():
    row = [
        r["Scenario"],
        f"+{r['Media Lift %']:.1f}%",
        f"[+{r['Media CI Low %']:.1f}%, +{r['Media CI High %']:.1f}%]" if r['Media CI Low %']>=0
            else f"[{r['Media CI Low %']:.1f}%, +{r['Media CI High %']:.1f}%]",
        f"+{r['Admissions Gained']:,.0f}",
        f"[{r['Gained CI Low']:+,.0f}, {r['Gained CI High']:+,.0f}]",
        f"{r['New Total Attendance']:,.0f}",
        f"+{r['Total Lift %']:.2f}%",
        f"[{r['Total CI Low %']:+.2f}%, {r['Total CI High %']:+.2f}%]",
        r["Confidence"],
    ]
    ws.append(row)
    rr = ws.max_row
    is_caveat = "crosses" in r["Confidence"]
    for j in range(1,10):
        cc = ws.cell(row=rr, column=j); cc.font=cell_font; cc.border=border
        cc.alignment = left if j==1 else center
        if is_caveat: cc.fill = caveat_fill

# footnote
fn = ws.max_row + 2
ws.cell(row=fn, column=1,
        value=("Notes: 'Media Lift %' is on media-attributable attendance; 'Total Lift %' is on real total admissions "
               "(media ≈ 23% of total, so lifts compress). Reshuffle is DIRECTIONAL — its CI crosses zero, i.e. the "
               "current allocation is already reasonably efficient and reshuffling may not measurably change attendance. "
               "Lifts apply only to validated media spend; geo-test LOB spend is held fixed. CI = 89% credible interval, "
               "conditional on the model — NOT the probability a scenario is enacted."))
ws.cell(row=fn, column=1).font = Font(name="Arial", italic=True, size=8, color="555555")
ws.cell(row=fn, column=1).alignment = Alignment(wrap_text=True, vertical="top"); ws.merge_cells(f"A{fn}:I{fn+3}")

widths = [22,12,20,18,24,20,12,20,26]
for j,w in enumerate(widths,1): ws.column_dimensions[chr(64+j)].width = w
ws.freeze_panes = "A5"

# ── Sheet 2: Attendance Structure ────────────────────────────────────────────
ws2 = wb.create_sheet("Attendance Structure")
ws2["A1"]="2025 Attendance Structure (reconciled to actual)"; ws2["A1"].font=Font(name="Arial",bold=True,size=12,color=NAVY); ws2.merge_cells("A1:C1")
for j,h in enumerate(["Component","Attendance","% of Total"],1):
    c=ws2.cell(row=3,column=j,value=h); c.font=hdr_font; c.fill=hdr_fill; c.alignment=center; c.border=border
struct=[("Baseline / non-media",controls_base,controls_base/actual_total*100),
        ("Validated media (scalable)",media_base,media_base/actual_total*100),
        ("TOTAL (actual 2025)",actual_total,100.0)]
for nm,val,pct in struct:
    ws2.append([nm,f"{val:,.0f}",f"{pct:.1f}%"])
    for j in range(1,4):
        cc=ws2.cell(row=ws2.max_row,column=j); cc.font=cell_font; cc.border=border; cc.alignment=left if j==1 else center
        if nm.startswith("TOTAL"): cc.font=Font(name="Arial",bold=True,size=10)
cr=ws2.max_row+2
ws2.cell(row=cr,column=1,value=(f"⚠ Contested attribution (inside total, NOT added): geo-test LOBs ~{geo_contested:,.0f}. "
        "The model credits these LOBs 0; we hypothesise this share of the total belongs to them rather than to "
        "baseline/validated media. Unproven — requires a geo-test. Total remains the actual figure above."))
ws2.cell(row=cr,column=1).font=Font(name="Arial",italic=True,size=8,color="C00000")
ws2.cell(row=cr,column=1).alignment=Alignment(wrap_text=True,vertical="top"); ws2.merge_cells(f"A{cr}:C{cr+3}")
ws2.cell(row=cr,column=1).fill=caveat_fill
for j,w in enumerate([30,18,12],1): ws2.column_dimensions[chr(64+j)].width=w

import shutil
_local_xlsx = "/tmp/FINAL_EXH_Scenario_Deliverable.xlsx"
xlsx_path = f"{DECOMP_PATH}FINAL_EXH_Scenario_Deliverable.xlsx"
wb.save(_local_xlsx)
shutil.copy2(_local_xlsx, xlsx_path)
os.remove(_local_xlsx)

# CSV backups
final_df.to_csv(f"{DECOMP_PATH}FINAL_scenario_outcomes.csv", index=False)
pd.DataFrame(struct, columns=["Component","Attendance","Pct_of_Total"]).to_csv(
    f"{DECOMP_PATH}FINAL_attendance_structure.csv", index=False)

print("✓ Exported:")
print(f"  {xlsx_path}")
print(f"  FINAL_scenario_outcomes.csv")
print(f"  FINAL_attendance_structure.csv")
print(f"  FINAL_scenario_both_frames.png")
print(f"  FINAL_attendance_structure.png")

In [0]:
# ── CELL: Export all chart data to a single Excel workbook ───────────────────
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as XLImage
import shutil, os, io

NAVY  = "002B5C"; WHITE = "FFFFFF"; LGREY = "F2F2F2"; AMBER = "FFF2CC"
GREEN = "2E8B57"; GOLD  = "F5A800"; RED   = "E31837"; GREY  = "9AA0A6"

hdr_font  = Font(name="Arial", bold=True, color=WHITE, size=10)
cell_font = Font(name="Arial", size=9)
sub_font  = Font(name="Arial", bold=True, color=NAVY, size=9)
center    = Alignment(horizontal="center", vertical="center", wrap_text=True)
left      = Alignment(horizontal="left",   vertical="center")
thin      = Side(style="thin", color="D0D0D0")
border    = Border(thin, thin, thin, thin)

TYPE_FILL = {
    "budget"   : PatternFill("solid", fgColor="DDEEFF"),
    "reshuffle": PatternFill("solid", fgColor=AMBER),
    "combined" : PatternFill("solid", fgColor="DDFFDD"),
    "uncertain": PatternFill("solid", fgColor="EEEEEE"),
}

def write_header(ws, title, subtitle, ncols):
    ws["A1"] = title
    ws["A1"].font = Font(name="Arial", bold=True, size=12, color=NAVY)
    ws.merge_cells(f"A1:{get_column_letter(ncols)}1")
    if subtitle:
        ws["A2"] = subtitle
        ws["A2"].font = Font(name="Arial", italic=True, size=9, color="555555")
        ws.merge_cells(f"A2:{get_column_letter(ncols)}2")
    return 4  # first data row

def style_header_row(ws, row, cols, fill_color=NAVY):
    fill = PatternFill("solid", fgColor=fill_color)
    for j, h in enumerate(cols, 1):
        c = ws.cell(row=row, column=j, value=h)
        c.font = hdr_font; c.fill = fill
        c.alignment = center; c.border = border

def style_data_row(ws, row, values, row_fill=None, ncols=None):
    fill = PatternFill("solid", fgColor=row_fill) if row_fill else PatternFill("solid", fgColor=LGREY)
    for j, v in enumerate(values, 1):
        c = ws.cell(row=row, column=j, value=v)
        c.font = cell_font; c.fill = fill
        c.border = border
        c.alignment = left if j == 1 else center

def set_col_widths(ws, widths):
    for j, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(j)].width = w

def get_row_fill(kind):
    if kind == "budget":    return "EEF5FF"
    if kind == "reshuffle": return "FFFBEA"
    if kind == "combined":  return "EFFFEF"
    return LGREY

wb = Workbook()
wb.remove(wb.active)  # remove default sheet

# ════════════════════════════════════════════════════════════════════════════
# TAB 1 — Scenario Summary (S1 diminishing returns + all scenarios CI)
# ════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("Scenario Summary")
r = write_header(ws,
    "EXH MMM — Scenario Summary (Media-Attributable Lift)",
    "89% CI = model-conditional credible interval on attendance lift. "
    "Confidence = CI width: High <4pp, Medium 4-8pp, Low >8pp.",
    9)

# S1 diminishing returns block
ws.cell(row=r, column=1, value="SCENARIO 1 — Budget Uplift (diminishing returns)").font = sub_font
ws.merge_cells(f"A{r}:I{r}"); r += 1

style_header_row(ws, r, [
    "Scenario", "Spend Increase %", "Annual Spend ($)",
    "Attendance Lift (att)", "Lift %",
    "CI Low %", "CI High %", "CI Width", "Confidence"
])
r += 1
for res in s1_results:
    style_data_row(ws, r, [
        res["label"], f"+{res['spend_increase_pct']:.0f}%",
        round(res["spend"], 0),
        round(res["lift"], 0), round(res["lift_pct"], 1),
        round(res["pct_lo"], 1), round(res["pct_hi"], 1),
        round(res["ci_width"], 1), res["confidence"],
    ], row_fill="EEF5FF")
    r += 1

r += 1  # blank row

# All scenarios block
ws.cell(row=r, column=1, value="ALL SCENARIOS — Annual Lift with 89% CI").font = sub_font
ws.merge_cells(f"A{r}:I{r}"); r += 1

style_header_row(ws, r, [
    "Scenario", "Type", "Annual Spend ($)",
    "Attendance Lift (att)", "Lift %",
    "CI Low %", "CI High %", "CI Width", "Confidence"
])
r += 1
for res in s1_results + [r2] + s3_results:
    style_data_row(ws, r, [
        res["label"], res.get("type", "—"),
        round(res["spend"], 0),
        round(res["lift"], 0), round(res["lift_pct"], 1),
        round(res["pct_lo"], 1), round(res["pct_hi"], 1),
        round(res["ci_width"], 1), res["confidence"],
    ], row_fill=get_row_fill(
        "budget" if "budget" in res["label"] else
        "reshuffle" if "Reshuffle" in res["label"] else "combined"
    ))
    r += 1

set_col_widths(ws, [24, 14, 18, 20, 10, 10, 10, 10, 22])
ws.freeze_panes = "A5"

# ════════════════════════════════════════════════════════════════════════════
# TAB 2 — Reconciled Scenario Outcomes (final_df — both frames)
# ════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("Reconciled Outcomes")
r = write_header(ws,
    "EXH MMM — Reconciled Scenario Outcomes (2025)",
    f"Actual total attendance: {actual_total:,.0f}  |  "
    f"Media = {MEDIA_SHARE*100:.1f}% of total  |  "
    f"Geo-test contested: {geo_contested:,.0f} (inside total, not additive)",
    11)

style_header_row(ws, r, [
    "Scenario", "Type",
    "Media Lift %", "Media CI Low %", "Media CI High %",
    "Admissions Gained", "Gained CI Low", "Gained CI High",
    "New Total Attendance", "Total Lift %", "Confidence"
])
r += 1
for _, row in final_df.iterrows():
    style_data_row(ws, r, [
        row["Scenario"], row["Type"],
        round(row["Media Lift %"], 1),
        round(row["Media CI Low %"], 1), round(row["Media CI High %"], 1),
        round(row["Admissions Gained"], 0),
        round(row["Gained CI Low"], 0), round(row["Gained CI High"], 0),
        round(row["New Total Attendance"], 0),
        round(row["Total Lift %"], 2),
        row["Confidence"],
    ], row_fill=get_row_fill(row["Type"] if "crosses" not in row["Confidence"] else "uncertain"))
    r += 1

set_col_widths(ws, [22, 12, 14, 14, 14, 18, 16, 16, 20, 12, 28])
ws.freeze_panes = "A5"

# ════════════════════════════════════════════════════════════════════════════
# TAB 3 — Attendance Structure
# ════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("Attendance Structure")
r = write_header(ws,
    "2025 Attendance Structure (reconciled to actual)",
    "Media share drives scenarios. Geo-test attribution is hypothesised inside total — not additive.",
    4)

style_header_row(ws, r, ["Component", "Attendance", "% of Total", "Notes"])
r += 1
struct_rows = [
    ("Baseline / Non-Media", round(controls_base, 0),
     f"{controls_base/actual_total*100:.1f}%", "Controls-driven attendance"),
    ("Validated Media", round(media_base, 0),
     f"{media_base/actual_total*100:.1f}%", "Scalable — scenarios operate on this"),
    ("TOTAL (actual 2025)", round(actual_total, 0),
     "100%", "Reconciled to actual reported attendance"),
    (f"⚠ Geo-test contested (inside total)", round(geo_contested, 0),
     f"{geo_contested/actual_total*100:.1f}%",
     "Hypothesised attribution — NOT additive to total above"),
]
for nm, val, pct, note in struct_rows:
    is_total   = "TOTAL" in nm
    is_geo     = "Geo" in nm
    fill_color = "FFF2CC" if is_geo else ("F2F2F2" if is_total else "FFFFFF")
    c_font     = Font(name="Arial", size=9,
                      bold=is_total,
                      italic=is_geo,
                      color="C05000" if is_geo else NAVY if is_total else "000000")
    ws.append([nm, val, pct, note])
    dr = ws.max_row
    fill = PatternFill("solid", fgColor=fill_color)
    for j in range(1, 5):
        cc = ws.cell(row=dr, column=j)
        cc.font = c_font; cc.fill = fill
        cc.border = border
        cc.alignment = left if j in (1, 4) else center

set_col_widths(ws, [30, 18, 14, 50])

# ════════════════════════════════════════════════════════════════════════════
# TAB 4 — LOB Spend by Scenario (one row per LOB, cols per scenario)
# ════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("LOB Spend by Scenario")
r = write_header(ws,
    "LOB Spend — Baseline vs Each Scenario (graft tilt split)",
    "Scenario spend = group-level scenario allocation × graft Final Share (grp). "
    "Δ = scenario minus baseline. Att Lift = Δ spend × att-per-$1 (linear approx at 2025 point).",
    4 + len(scenarios) * 3)

# scenarios here uses (label, alloc) tuples — rebuild with lift results
scenarios_with_lift = (
    [(r_["label"],
      {c: baseline_spend[c] * (1 + r_["spend_increase_pct"] / 100) for c in ACTIVE_COMBOS},
      r_)
     for r_ in s1_results]
    + [("Reshuffle (10% cap)", s2_alloc, r2)]
    + [(r_["label"],
        {c: reshuffle({c2: baseline_spend[c2] * (1 + r_["spend_increase_pct"] / 100)
                       for c2 in ACTIVE_COMBOS})[c]
         for c in ACTIVE_COMBOS},
        r_)
       for r_ in s3_results]
)

# Fixed cols header (spans 2 rows)
fixed_hdrs = ["Funnel", "LOB", "Baseline Spend ($)"]
for j, h in enumerate(fixed_hdrs, 1):
    c = ws.cell(row=r, column=j, value=h)
    c.font = hdr_font
    c.fill = PatternFill("solid", fgColor=NAVY)
    c.alignment = center; c.border = border
    ws.merge_cells(start_row=r, start_column=j, end_row=r+1, end_column=j)

SCEN_HEADER_FILLS = {
    "budget"   : "DDEEFF",
    "reshuffle": "FFF2CC",
    "combined" : "DDFFDD",
}
col_cursor = 4
for scen_label, _, lift_r in scenarios_with_lift:
    kind      = ("budget" if "budget" in scen_label
                 else "reshuffle" if "Reshuffle" in scen_label else "combined")
    fill_hex  = SCEN_HEADER_FILLS[kind]
    s_fill    = PatternFill("solid", fgColor=fill_hex)
    merge_end = col_cursor + 2
    lr        = lift_r
    hval      = (f"{scen_label}  |  +{lr['lift_pct']:.1f}%  "
                 f"CI:[+{lr['pct_lo']:.1f}%,+{lr['pct_hi']:.1f}%]  "
                 f"Conf:{lr['confidence']}")
    c = ws.cell(row=r, column=col_cursor, value=hval)
    c.font = Font(name="Arial", bold=True, color=NAVY, size=9)
    c.fill = s_fill; c.alignment = center; c.border = border
    ws.merge_cells(start_row=r, start_column=col_cursor,
                   end_row=r, end_column=merge_end)
    for j, sub in enumerate(["Scenario Spend ($)", "Δ ($)", "Att Lift"], col_cursor):
        sc = ws.cell(row=r+1, column=j, value=sub)
        sc.font = sub_font; sc.fill = s_fill
        sc.alignment = center; sc.border = border
    col_cursor = merge_end + 1

r += 2  # move past the two header rows

FUNNEL_FILLS = {
    "Awareness"    : "EEF3FA",
    "Consideration": "FDEAEA",
    "Conversion"   : "FFFBEA",
}

# Build att-per-$1 map
fl_rate_map = {}
for _, drow in decomp_valid.iterrows():
    key = (drow["Funnel"], drow["LOB"])
    if pd.notna(drow["Att per $1"]) and drow["Annual Spend"] > 0:
        fl_rate_map.setdefault(key, []).append(
            (float(drow["Att per $1"]), float(drow["Annual Spend"]))
        )
fl_rate_lob = {}
for key, vals in fl_rate_map.items():
    total_sp = sum(v[1] for v in vals)
    fl_rate_lob[key] = sum(v[0]*v[1] for v in vals) / total_sp if total_sp > 0 else 0.0

for funnel, lob in lob_keys:
    base_sp  = baseline_lob_spend.get((funnel, lob), 0.0)
    rate     = fl_rate_lob.get((funnel, lob), 0.0)
    row_fill = PatternFill("solid", fgColor=FUNNEL_FILLS.get(funnel, "FFFFFF"))
    data_row = [funnel, lob, round(base_sp, 0)]
    for scen_label, group_alloc, _ in scenarios_with_lift:
        scen_sp  = split_to_lobs(group_alloc).get((funnel, lob), 0.0)
        delta_sp = scen_sp - base_sp
        data_row += [round(scen_sp, 0), round(delta_sp, 0), round(delta_sp * rate, 0)]
    ws.append(data_row)
    dr = ws.max_row
    for j in range(1, len(data_row) + 1):
        cc = ws.cell(row=dr, column=j)
        cc.font = cell_font; cc.fill = row_fill
        cc.border = border
        cc.alignment = left if j <= 2 else center

ws.column_dimensions["A"].width = 16
ws.column_dimensions["B"].width = 30
ws.column_dimensions["C"].width = 18
col_cursor = 4
for _ in scenarios_with_lift:
    for w in [18, 14, 14]:
        ws.column_dimensions[get_column_letter(col_cursor)].width = w
        col_cursor += 1
ws.freeze_panes = "C4"
# ════════════════════════════════════════════════════════════════════════════
# IMAGE TABS — one per chart
# ════════════════════════════════════════════════════════════════════════════
def embed_img(ws, path, anchor, w=900):
    if os.path.exists(path):
        img = XLImage(path)
        img.width = w; img.height = int(w * 9 / 16)
        ws.add_image(img, anchor)
    else:
        ws[anchor] = f"Image not found: {os.path.basename(path)}"

chart_tabs = [
    ("Scenario Outcomes",      "scenario_outcomes.png"),
    ("Media Lift",             "FINAL_media_lift.png"),
    ("Attendance Stacked",     "FINAL_attendance_stacked.png"),
    ("Attendance Structure",   "FINAL_attendance_structure.png"),
    ("Both Frames",            "FINAL_scenario_both_frames.png"),
]

# Per-scenario LOB spend charts
for scen_label, _, _ in scenarios_with_lift:
    safe = scen_label.replace(" ", "_").replace("%", "pct").replace("+", "plus").replace("/", "_")
    fname = f"I2_B10_LOB_Spend_{safe}.png"
    tab   = scen_label[:28].strip()
    chart_tabs.append((tab, fname))

for tab_name, fname in chart_tabs:
    ws = wb.create_sheet(tab_name)
    ws["A1"] = tab_name
    ws["A1"].font = Font(name="Arial", bold=True, size=12, color=NAVY)
    embed_img(ws, os.path.join(OUT_PATH, fname), "A3")
# ════════════════════════════════════════════════════════════════════════════
# Save
# ════════════════════════════════════════════════════════════════════════════
_tmp      = "/tmp/FINAL_EXH_Scenario_Charts_Data.xlsx"
xlsx_path = os.path.join(OUT_PATH, "FINAL_EXH_Scenario_Charts_Data.xlsx")
wb.save(_tmp)
shutil.copy2(_tmp, xlsx_path)
os.remove(_tmp)
print(f"✓ Saved: {xlsx_path}")
print("  Tabs: Scenario Summary | Reconciled Outcomes | Attendance Structure | LOB Spend by Scenario")